# Research data supporting "Is fixed node diffusion quantum Monte Carlo reproducible?"

This notebook accompanies the paper: **Reproducibility of fixed-node diffusion Monte Carlo across diverse community codes: The case of water-methane dimer**. It can be found on Github at: https://github.com/zenandrea/DMC-reproducibility and also on [Colab](https://colab.research.google.com/github/zenandrea/DMC-reproducibility/blob/main/analysis.ipynb)

### Abstract

Fixed-node quantum diffusion Monte Carlo (FN-DMC) is a widely-trusted many-body method for solving the Schr\"{o}dinger equation, known for its reliable predictions of material and molecular properties. Furthermore, its excellent scalability with system complexity and near-perfect utilization of computational power makes FN-DMC ideally positioned to leverage new advancements in computing to address increasingly complex scientific problems. However, demonstrating reproducibility across different FN-DMC code implementations remains an outstanding challenge. This difficulty stems from the diverse array of DMC algorithms and wave-function implementations, compounded by the method's inherent stochastic nature. This study represents a community-wide effort to address the titular question, affirming that: Yes, FN-DMC is reproducible. Using the water-methane dimer as the canonical test case, we identified the treatment of the non-locality of pseudopotentials as the primary source of discrepancies between different implementations. In particular, we demonstrate that while the locality approximation (LA) leads to considerable variability in results, reliable and reproducible predictions can be achieved by employing the T-move (TM), the determinant locality approximation (DLA), or the determinant locality T-move (DTM) schemes. This work lays the foundation to establish accurate and reproducible FN-DMC estimates for all future studies across its countless applications in materials science, physics, chemistry, and biology.

In [1]:
# Check if we are in Google Colab environment
try:
    import google.colab
    IN_COLAB = True
    usetex = False
except:
    import os
    IN_COLAB = False
    if os.path.expanduser('~') == '/home/shixubenjamin':
        usetex = True
    else:
        usetex = True #False

# If in Google Colab, install the necessary data and set up the necessary environment
if IN_COLAB == True:
    !rm -rf /content/DMC-reproducibility-main /content/main.zip
    !wget https://github.com/zenandrea/DMC-reproducibility/archive/refs/heads/main.zip
    !unzip /content/main.zip
    %pwd
    %cd /content/DMC-reproducibility-main



In [2]:
print(f"IN_COLAB = {IN_COLAB}")
print(f"usetex = {usetex}")

IN_COLAB = False
usetex = True


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.optimize import curve_fit
from uncertainties import ufloat

In [4]:
from Scripts.myfit import *
from Scripts.myfunctions import *
#from Scripts.mysetup import *
#from Scripts.jup_plot import *

In [5]:
# Hatree to eV
Hartree_to_eV = 27.211386245988
# Hartree to meV
Ha2meV = Hartree_to_eV * 1000

In [6]:
def texfalse_import():
    plt.rcParams.update(
        {
            "font.family": "serif",  # use serif/main font for text elements
            "font.size": 9,
            "text.usetex": False,  # use inline math for ticks
        }
    )

def textrue_import():
    import matplotlib as mpl
    mpl.use("pgf")
    plt.rcParams.update({
        "font.family": "serif",  # use serif/main font for text elements
        "font.size": 9,
        "text.usetex": True,     # use inline math for ticks
        "pgf.rcfonts": False,    # don't setup fonts from rc parameters
        "pgf.preamble":"\n".join([
            r"\usepackage{amsmath}",            # load additional packages
            r"\usepackage{amssymb}",   # unicode math setup
            r"\usepackage[mathrm=sym]{unicode-math}",  # serif font via preamble
            r"\setmainfont[BoldFont={FiraSans-SemiBold.otf}]{FiraSans-Regular.otf}",
            r"\setmathfont{FiraMath-Regular.otf}",
            #r"\setmathfont[version=bold]{FiraMath-Bold.otf}",
            r"\newcommand{\minus}{\scalebox{0.5}[1.0]{$-$}}" # serif font via preamble
        ])
    })

# Dictionary of colors for the color scheme in our plots
color_dict = {
    "red": "#e6194b",
    "green": "#3cb44b",
    "yellow": "#ffe119",
    "blue": "#4363d8",
    "orange": "#f58231",
    "purple": "#911eb4",
    "cyan": "#42d4f4",
    "magenta": "#f032e6",
    "lime": "#bfef45",
    "pink": "#fabed4",
    "teal": "#469990",
    "lavendar": "#dcbeff",
    "brown": "#9A6324",
    "beige": "#fffac8",
    "maroon": "#800000",
    "mint": "#aaffc3",
    "olive": "#808000",
    "apricot": "#ffd8b1",
    "navy": "#000075",
    "grey": "#a9a9a9",
    "white": "#ffffff",
    "black": "#000000",
}

# Colors to cycle through for our plots
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=[
        "#4363d8",
        "#e6194B",
        "#3cb44b",
        "#f58231",
        "#ffe119",
        "#911eb4",
        "#42d4f4",
        "#f032e6",
        "#bfef45",
        "#fabed4",
        "#469990",
        "#dcbeff",
        "#9A6324",
        "#fffac8",
        "#800000",
        "#aaffc3",
        "#808000",
        "#ffd8b1",
        "#000075",
        "#a9a9a9",
        "#ffffff",
        "#000000",
    ]
)

In [7]:
if usetex:
    textrue_import()
else:
    texfalse_import()

## CCSDT(Q) reference

In [8]:
cc_bind_ene_cp = {x:0 for x in ['DZ','TZ','QZ','5Z','CBS(DZ/TZ)','CBS(TZ/QZ)','CBS(QZ/5Z)']}
cc_bind_ene_nocp = {x:0 for x in ['DZ','TZ','QZ','5Z','CBS(DZ/TZ)','CBS(TZ/QZ)','CBS(QZ/5Z)']}

ene_hf_cp = []
ene_ccsdt_cp = []
ene_hf_nocp = []
ene_ccsdt_nocp = []

# Read the calculation outputs from MRCC. We calculate using counterpoise correction (cp) and without counterpoise correction (nocp) at the CBS(QZ/5Z) for CCSD(T) and DZ level for CCSDT(Q). Final results are the average of cp and nocp.
for i in ['DZ','TZ','QZ','5Z']:
    ene_hf_cp +=  [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_cp/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_cp/mrcc.out')['scf_energy'])*1000]
    ene_hf_nocp += [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_nocp/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_nocp/mrcc.out')['scf_energy'])*1000]
    ene_ccsdt_cp += [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_cp/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_cp/mrcc.out')['ccsd(t)_corr_energy'])*1000]
    ene_ccsdt_nocp += [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_nocp/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_nocp/mrcc.out')['ccsd(t)_corr_energy'])*1000]

    cc_bind_ene_cp[i] = (read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_cp/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_cp/mrcc.out')['energy'])*1000
    cc_bind_ene_nocp[i] = (read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_nocp/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_nocp/mrcc.out')['energy'])*1000

print('> CCSD(T) CBS w/ & w/o CP')
# Let's get CCSD(T) CBS(DZ/TZ)
cc_bind_ene_cp['CBS(DZ/TZ)']   = get_cbs(ene_hf_cp[0],  ene_ccsdt_cp[0],  ene_hf_cp[1],  ene_ccsdt_cp[1],  X=2,Y=3,family='acc',output=True)[-1]
cc_bind_ene_nocp['CBS(DZ/TZ)'] = get_cbs(ene_hf_nocp[0],ene_ccsdt_nocp[0],ene_hf_nocp[1],ene_ccsdt_nocp[1],X=2,Y=3,family='acc',output=True)[-1]

# Let's get CCSD(T) CBS(TZ/QZ)
cc_bind_ene_cp['CBS(TZ/QZ)'] = get_cbs(ene_hf_cp[1],ene_ccsdt_cp[1],ene_hf_cp[2],ene_ccsdt_cp[2],X=3,Y=4,family='acc',output=True)[-1]
cc_bind_ene_nocp['CBS(TZ/QZ)'] = get_cbs(ene_hf_nocp[1],ene_ccsdt_nocp[1],ene_hf_nocp[2],ene_ccsdt_nocp[2],X=3,Y=4,family='acc',output=True)[-1]

# Let's get CCSD(T) CBS(QZ/5Z)
cc_bind_ene_cp['CBS(QZ/5Z)'] = get_cbs(ene_hf_cp[2],ene_ccsdt_cp[2],ene_hf_cp[3],ene_ccsdt_cp[3],X=4,Y=5,family='acc',output=True)[-1]
cc_bind_ene_nocp['CBS(QZ/5Z)'] = get_cbs(ene_hf_nocp[2],ene_ccsdt_nocp[2],ene_hf_nocp[3],ene_ccsdt_nocp[3],X=4,Y=5,family='acc',output=True)[-1]

print('> Get CCSDT(Q) DZ') 
# Let's get CCSDT(Q) DZ
ene_ccsdtq_cp = 0
ene_ccsdtq_nocp = 0

ene_ccsdtq_cp   = (read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/mw/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/m_cp/mrcc.out')['ccsdtq_corr_energy']   - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/w_cp/mrcc.out')['ccsdtq_corr_energy']  )*1000
ene_ccsdtq_nocp = (read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/mw/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/m_nocp/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/w_nocp/mrcc.out')['ccsdtq_corr_energy'])*1000

print('> Evaluate Eint as CCSD(T)/CBS - CCSD(T)/DZ + CCSDT(Q)/DZ ')
final_ccsdtq_cp   = ene_ccsdtq_cp   - ene_ccsdt_cp[0]   + cc_bind_ene_cp['CBS(QZ/5Z)']
final_ccsdtq_nocp = ene_ccsdtq_nocp - ene_ccsdt_nocp[0] + cc_bind_ene_nocp['CBS(QZ/5Z)']

final_ccsdt_ref = (cc_bind_ene_cp['CBS(QZ/5Z)'] + cc_bind_ene_nocp['CBS(QZ/5Z)']) / 2
print(f'Final CCSD(T) Interaction energy: {final_ccsdt_ref:.2f} meV')

final_ccsdtq_ref = (final_ccsdtq_cp + final_ccsdtq_nocp) / 2
print(f'Final CCSDT(Q) Interaction energy: {final_ccsdtq_ref:.2f} meV')

> CCSD(T) CBS w/ & w/o CP
CBS(2/3)  HF: 8.813730103 Corr: -36.143731834 Tot: -27.330001731
CBS(2/3)  HF: 9.681718339 Corr: -36.849591616 Tot: -27.167873277
CBS(3/4)  HF: 8.543110165 Corr: -35.243642528 Tot: -26.700532363
CBS(3/4)  HF: 8.267090833 Corr: -34.548930349 Tot: -26.281839515
CBS(4/5)  HF: 8.527723648 Corr: -35.074195251 Tot: -26.546471603
CBS(4/5)  HF: 8.625516587 Corr: -34.844709809 Tot: -26.219193223
> Get CCSDT(Q) DZ
> Evaluate Eint as CCSD(T)/CBS - CCSD(T)/DZ + CCSDT(Q)/DZ 
Final CCSD(T) Interaction energy: -26.38 meV
Final CCSDT(Q) Interaction energy: -26.62 meV


## Analysis of DMC code data

In [9]:
# Defining variables and colours

systems = [ 'm', 'w', 'mw', 'm---w' ]
methods = [ 'LA', 'TM', 'DLA', 'DLTM' ]

# Function to rename DLTM to DTM
method_rename = lambda x: 'DTM' if x=='DLTM' else x 


# Define dictionary with the methods implemented within each code (alongside folder location) and the corresponding label and color
dmc_code_info ={
    'Amolqc': {'methods_and_folders': {x: 'Amolqc' for x in ['DLA']}, 
               'label': 'Amolqc', 
               'color': '#e6194B', 
               'discretization': 'tau',
               'degreefit' : 2,
              },
    'CASINO': {'methods_and_folders': {x: 'CASINO' for x in['LA','TM','DLA','DLTM']}, 
               'label': 'CASINO', 
               'color': '#f58231', 
               'discretization': 'tau',
               'degreefit' : 2,
               'degreefit_w' : 3,
               'degreefit_mw' : 3,
               'degreefit_m---w' : 3,
              },
    'CHAMP-EU': {'methods_and_folders': {x:'CHAMP-EU' for x in ['LA','TM']} , 
                 'label': 'CHAMP-EU', 
                 'color': '#ffe119', 
                 'discretization': 'tau',
                 'degreefit' : 2,
                 'degreefit_w' : 3,
                 'degreefit_mw' : 3,
                 'degreefit_m---w' : 3,
                },
    'CHAMP-US': {'methods_and_folders': {x:'CHAMP-US' for x in ['LA','TM']}, 
                 'label': 'CHAMP-US', 
                 'color': '#bfef45', 
                 'discretization': 'tau',
                 'degreefit' : 2,
                 'degreefit_m_LA' : 3,
                 'degreefit_w_LA' : 3,
                 'degreefit_mw_LA' : 3,
                 'degreefit_m---w_LA' : 3,
                },
    'CMQMC': {'methods_and_folders': {x:'CMQMC' for x in ['LA','TM','DLA','DLTM']}, 
              'label': 'CMQMC', 
              'color': '#3cb44b', 
              'discretization': 'tau',
              'degreefit' : 2,
              'degreefit_m_LA' : 3,
              'degreefit_m_DLA' : 3,
              'degreefit_w_LA' : 3,
              'degreefit_w_DLA' : 3,
              'degreefit_mw' : 3,
              'degreefit_m---w' : 3,
             },
    'PyQMC': {'methods_and_folders': {x:'PyQMC' for x in ['LA','TM','DLA']}, 
              'label': 'PyQMC', 
              'color': '#42d4f4', 
              'discretization': 'tau',
              'degreefit' : 2,
              'degreefit_w' : 3,
              'degreefit_mw' : 3,
              'degreefit_m---w' : 3,
             },
    'QMC=Chem': {'methods_and_folders': {x:'QMC=Chem' for x in ['DLA']}, 
                 'label': 'QMC=Chem', 
                 'color': '#4363d8', 
                 'discretization': 'tau',
                 'degreefit' : 3,
                },
    'QMCPACK': {'methods_and_folders': {x: 'QMCPACK' for x in['LA','TM','DLA','DLTM']}, 
                'label': 'QMCPACK', 
                'color': '#911eb4', 
                'discretization': 'tau',
                'degreefit' : 2,
                'degreefit_m_LA' : 3,
                'degreefit_w' : 3,
                'degreefit_mw' : 3,
                'degreefit_m---w' : 3,
               },
    'QMeCha': {'methods_and_folders': {x:'QMeCha' for x in ['LA','DLA']}, 
               'label': 'QMeCha', 
               'color': '#f032e6', 
               'discretization': 'tau',
               'degreefit' : 2,
               'degreefit_m' : 3, 
               'degreefit_w' : 3, 
               'degreefit_mw' : 3, 
               'degreefit_m---w' : 3, 
              },
    'QWalk': {'methods_and_folders': {x:'QWalk_noncov' for x in ['LA','TM','DLA']}, 
              'label': 'QWalk', 
              'color': '#800000', 
              'discretization': 'tau',
              'degreefit' : 2,
              'degreefit_m_LA' : 3,
              'degreefit_w' : 3,
              'degreefit_mw' : 3,
              'degreefit_m---w' : 3,
             },
    'TurboRVB-DMC': {'methods_and_folders': {x:'TurboRVB-DMC' for x in ['TM']}, 
                     'label': 'TurboRVB', 
                     'color': '#469990', 
                     'discretization': 'tau',
                     'degreefit' : 3,
                    },
    'TurboRVB-LRDMC': {'methods_and_folders': {x:'TurboRVB-LRDMC' for x in ['TM','DLA','DLTM']}, 
                       'label': 'TurboRVB (LRDMC)', 
                       'color': '#808000', 
                       'discretization': 'a',
                       'degreefit' : 2,
                       'degreefit_m---w_TM' : 3,
                       'degreefit_DLTM' : 3,
                      }
}

method_codes_dict = {
    'LA'  : [code for code in dmc_code_info.keys() if 'LA'   in dmc_code_info[code]['methods_and_folders'].keys()],
    'TM'  : [code for code in dmc_code_info.keys() if 'TM'   in dmc_code_info[code]['methods_and_folders'].keys()],
    'DLA' : [code for code in dmc_code_info.keys() if 'DLA'  in dmc_code_info[code]['methods_and_folders'].keys()],
    'DLTM': [code for code in dmc_code_info.keys() if 'DLTM' in dmc_code_info[code]['methods_and_folders'].keys()]
}

code_to_num_dict = {x: index for index, x in enumerate(dmc_code_info.keys())}
code_to_num_dict

{'Amolqc': 0,
 'CASINO': 1,
 'CHAMP-EU': 2,
 'CHAMP-US': 3,
 'CMQMC': 4,
 'PyQMC': 5,
 'QMC=Chem': 6,
 'QMCPACK': 7,
 'QMeCha': 8,
 'QWalk': 9,
 'TurboRVB-DMC': 10,
 'TurboRVB-LRDMC': 11}

In [10]:
# Initialize dictionary to store the data. 
# The dictionary is structured such that the keys are the codes and the values are dictionaries 
# with keys 'raw', 'fit' and 'final'. 
# 'raw' key contains the raw data from the calculations containing the values at the calculated timesteps, 
# 'fit' contains the fitted data and 
# 'final' contains the final interaction energy extrapolated to the zero time step limit.

#reanalyze_data = False 
reanalyze_data = True

# max tau for fit
taumax = 0.1

print(f"reanalyze_data = {reanalyze_data}")
print(f"taumax = {taumax}")


l_Eint = []
dl_Etot = { s:[] for s in systems }

eq2system = lambda x:'m---w' if x == 'mw_far' else x
system2eq = lambda x:'mw_far' if x == 'm---w' else x

# Head Table fit
print(' '*20,'taumin Ndata deg RCS  RMSD  WRMSD     Etau0        Etaumin      Delta   expDelta')

if reanalyze_data:
    dmc_code_data = {
        code: {quantity: {'raw': {}, 
                          'fit': {}, 
                          'final': {}} 
               for quantity in ['total_energy',
                                'interaction_energy', 
                                'interaction_energy_far',
                                'size_consistency_error']} 
        for code in dmc_code_info.keys()}
    final_interaction_energy = {code: {} for code in dmc_code_info.keys()}

    for code in dmc_code_info.keys():
        
        for method, method_folder in dmc_code_info[code]["methods_and_folders"].items():
            print( f'{code.ljust(15)} {method_rename(method)}' )
            for system in systems:
                key = f'{system}_{method}'
                # Read in the raw total energy data from the csv files
                raw_data = pd.read_csv(
                    f'Codes_Data/{method_folder}/energy_{key}.csv',
                    decimal='.', header=None, 
                    names=[dmc_code_info[code]["discretization"], 'ene','err'], 
                    index_col=dmc_code_info[code]["discretization"])
                dmc_code_data[code]['total_energy']['raw'][key] = raw_data

            # Calculate the interaction energy, interaction energy (far) and size consistency error from the total energy of the methane-water dimer, methane, water and the separate methane-water system.

            mw = dmc_code_data[code]['total_energy']['raw'][f'mw_{method}']
            m = dmc_code_data[code]['total_energy']['raw'][f'm_{method}']
            w = dmc_code_data[code]['total_energy']['raw'][f'w_{method}']
            mw_far = dmc_code_data[code]['total_energy']['raw'][f'm---w_{method}']

            # interaction_energy : AB - A - B
            dmc_code_data[code]['interaction_energy']['raw'][method] = pd.DataFrame([
                Ha2meV*(mw.ene-m.ene-w.ene), 
                Ha2meV*(mw.err**2+m.err**2+w.err**2)**0.5
            ]).T.dropna()
            
            # interaction_energy_far : AB - A-far-B
            dmc_code_data[code]['interaction_energy_far']['raw'][method] = pd.DataFrame([
                Ha2meV*(mw.ene - mw_far.ene), 
                Ha2meV*(mw.err**2+mw_far.err**2)**0.5
            ]).T.dropna()

            # size_consistency_error : A-far-B - A - B
            dmc_code_data[code]['size_consistency_error']['raw'][method] = pd.DataFrame([
                Ha2meV*(mw_far.ene-m.ene-w.ene), 
                Ha2meV*(mw_far.err**2+m.err**2+w.err**2)**0.5
            ]).T.dropna()

            # Fit the the interaction energy and total energy quantities
            for energy_quantity in ['mw','mw_far','m','w','interaction_energy','interaction_energy_far','size_consistency_error']:
                
                system = eq2system(energy_quantity)
                if system in systems:
                    fitting_data = dmc_code_data[code]['total_energy']['raw'][f'{system}_{method}']
                    if f'degreefit_{system}_{method}' in dmc_code_info[code].keys():
                        degree = dmc_code_info[code][f'degreefit_{system}_{method}']
                    elif f'degreefit_{system}' in dmc_code_info[code].keys():
                        degree = dmc_code_info[code][f'degreefit_{system}']
                    elif f'degreefit_{method}' in dmc_code_info[code].keys():
                        degree = dmc_code_info[code][f'degreefit_{method}']
                    else:
                        degree = dmc_code_info[code]['degreefit']
                elif energy_quantity in ['interaction_energy','interaction_energy_far','size_consistency_error']:
                    fitting_data = dmc_code_data[code][energy_quantity]['raw'][method]
                    if f'degreefit_int_{method}' in dmc_code_info[code].keys():
                        degree = dmc_code_info[code][f'degreefit_int_{method}']
                    elif f'degreefit_int' in dmc_code_info[code].keys():
                        degree = dmc_code_info[code][f'degreefit_int']
                    else:
                        degree = dmc_code_info[code]['degreefit']

                fitting_data['x'] = x_axis( fitting_data, dmc_code_info[code]['discretization'] )
                fitting_data_trunc = fitting_data[ fitting_data.x <= taumax ].dropna()

                xdata = fitting_data_trunc.x.to_numpy()
                ydata = fitting_data_trunc.ene.to_numpy()
                sigma = fitting_data_trunc.err.to_numpy()

                try:
                    #print(f'Fitting {energy_quantity}')
                    coeffs, coeffs_err, cov, chi_squared_reduced, RMSR, WRMSR, fitting_function = fit_polynomial( 
                        xdata, ydata, sigma, degree )
    
                    # Getting the quantities in the limit of timestep = 0
                    lim_E = coeffs[-1]
                    lim_E_sigma = coeffs_err[-1]
                    final = [ lim_E, lim_E_sigma ]
                    lim_Eu = ufloat( lim_E, lim_E_sigma )
    
                    # Fitting function with uncertainty
                    space_pts = 101
                    x_fit = np.linspace( 0, taumax, space_pts)
                    E_fit = fitting_function( x_fit )
                    # get the variance of the Y value, with Y = a + b X + c X^2 + ... 
                    # having the covariance matrix 
                    # ( cov has to be reindexed, because coeffs are in decreasing order of degree )
                    var_y = fun_var_y( x_fit, cov[::-1,::-1] )
                    # standard error on Y is the square root of the variance of Y
                    err_y = np.sqrt( var_y )
                    E_fit_lower_lim = E_fit - err_y
                    E_fit_upper_lim = E_fit + err_y
                    
                    # dictionary with fitting and extrapolated results
                    d_fitting = {
                        'x': xdata, 
                        'E': ydata , 
                        'E_err': sigma, 
                        'x_fit': x_fit, 
                        'E_fit': E_fit, 
                        'E_fit_lower_lim': E_fit_lower_lim, 
                        'E_fit_upper_lim': E_fit_upper_lim,
                        'degree' : degree,
                        'coeffs' : coeffs,
                        'coeffs_err' : coeffs_err,
                        'cov' : cov,
                        'chi_squared_reduced' : chi_squared_reduced, 
                        'RMSR' : RMSR,
                        'WRMSR' : WRMSR,
                    }

                    # Data for Tables about the fit written in dictionary 
                    tau_min = np.min(xdata)
                    y_smallest_tau = ydata[np.argmin(xdata)]
                    sigma_smallest_tau = sigma[np.argmin(xdata)]
                    Etaumin = ufloat( y_smallest_tau, sigma_smallest_tau )
                    Etau0   = lim_Eu
                    Delta = Etaumin - Etau0
                    lim_dEdtau = ufloat( coeffs[-2], coeffs_err[-2] )
                    DDelta = Delta - tau_min * lim_dEdtau
                    d_tab = {
                        'Code'     : code,
                        'Method'   : method_rename(method),
                        'tau_min'  : tau_min,
                        'Etaumin'  : Etaumin,
                        'Etau0'    : Etau0,
                        'dE/dtau'  : lim_dEdtau,
                        'Delta'    : Delta,
                        'DDelta'   : DDelta,
                        'degree'   : degree,
                        'Ndata'    : len(ydata),
                        'RCS'      : chi_squared_reduced,
                        'RMSR'     : RMSR,
                        'WRMSR'    : WRMSR,
                    }

                    system = eq2system(energy_quantity)
                    if system in systems:
                        dmc_code_data[code]['total_energy']['fit'][f'{system}_{method}'] = d_fitting
                        dmc_code_data[code]['total_energy']['final'][f'{system}_{method}'] = final
                        dl_Etot[system].append(d_tab)
                    elif energy_quantity in ['interaction_energy','interaction_energy_far','size_consistency_error']:
                        dmc_code_data[code][energy_quantity]['fit'][f'{method}'] = d_fitting
                        dmc_code_data[code][energy_quantity]['final'][method] = final
                        if energy_quantity == 'interaction_energy':
                            final_interaction_energy[code][method] = final
                            l_Eint.append(d_tab)
    
                    # print fitted data
                    print(f'{energy_quantity.rjust(24)} {tau_min:5.4f} {len(ydata)} {degree} {chi_squared_reduced:-5.2f} {RMSR:-6.2g} {WRMSR:-6.2g}',
                          f'{Etau0:-14.1uS} {Etaumin:-14.1uS} {Delta:-14.1uS} {tau_min*ufloat(coeffs[-2],coeffs_err[-2]):-14.1uS}' )
                except:
                    print( f'{energy_quantity.rjust(20)} Error fitting' )
                
    
    final_total_energy = {
        code: { 
            method: {
                system: dmc_code_data[code]['total_energy']['final'][f'{system}_{method}'] for system in ['mw', 'm', 'w']
            } for method in dmc_code_info[code]['methods_and_folders'].keys()
        } for code in dmc_code_info.keys()}
    
    np.save('Codes_Data/dmc_code_data.npy',dmc_code_data)
    np.save('Codes_Data/final_interaction_energy.npy',final_interaction_energy)
    np.save('Codes_Data/final_total_energy.npy',final_total_energy)

    df_Eint = pd.DataFrame( l_Eint )
    df_Eint.to_csv( 'Tables/df_Eint.csv', index=False)
    dfs_Etot = {
        system : pd.DataFrame( dl_Etot[system] ) for system in systems if dl_Etot[system]
    }
    for system, df in dfs_Etot.items():
        df.to_csv( f'Tables/df_Etot_{system}.csv', index=False)
else:
    dmc_code_data = np.load('Codes_Data/dmc_code_data.npy',allow_pickle=True).item()
    final_interaction_energy = np.load('Codes_Data/final_interaction_energy.npy',allow_pickle=True).item()
    final_total_energy = np.load('Codes_Data/final_total_energy.npy',allow_pickle=True).item()
    df_Eint = pd.read_csv('Tables/df_Eint.csv')
    dfs_Etot = {}
    for system in systems:
        dfs_Etot[system] = pd.read_csv(f'Tables/df_Etot_{system}.csv')

#print(df_Eint)

reanalyze_data = True
taumax = 0.1
                     taumin Ndata deg RCS  RMSD  WRMSD     Etau0        Etaumin      Delta   expDelta
Amolqc          DLA
                      mw 0.0025 5 2  0.80 1.7e-05 1.7e-05   -25.33190(3)   -25.33176(3)     0.00015(4)     0.00016(1)
                  mw_far 0.0025 4 2  0.01 1.2e-06 1.2e-06  -25.330842(4)   -25.33074(3)     0.00010(3)    0.000124(2)
                       m 0.0025 5 2  1.31 2.2e-05 2.1e-05    -8.08450(4)    -8.08445(3)       5(5)e-05       5(1)e-05
                       w 0.0025 5 2  0.52 1.4e-05 1.4e-05   -17.24638(2)   -17.24621(3)     0.00017(4)    0.000160(8)
      interaction_energy 0.0025 5 2  0.16   0.36   0.36       -27.9(6)         -30(1)          -2(2)        -1.4(2)
interaction_energy_far Error fitting
size_consistency_error Error fitting
CASINO          LA
                      mw 0.0050 5 3  0.72 1.1e-05 1.1e-05   -25.32427(4)   -25.32404(3)     0.00023(5)     0.00026(3)
                  mw_far 0.0050 5 3  0.01 1.

## Fitting quality of DMC energies and extrapolation

See cases with a large reduced-chi-squared

### Eint

In [11]:
df_Eint

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
0,Amolqc,DLA,0.0025,-29.8+/-1.4,-27.9+/-0.6,(-5.5+/-0.8)e+02,-1.9+/-1.5,-0.5+/-1.5,2,5,0.162907,0.359294,0.358451
1,CASINO,LA,0.0050,-33.3+/-1.0,-32.8+/-1.2,(-9+/-9)e+01,-0.6+/-1.6,-0.1+/-1.7,2,5,1.149731,0.763865,0.758412
2,CASINO,TM,0.0050,-27.2+/-1.1,-27.2+/-0.9,(1.8+/-0.7)e+02,0.0+/-1.5,-0.9+/-1.5,2,5,0.670597,0.570726,0.578798
3,CASINO,DLA,0.0025,-30.3+/-1.4,-31.2+/-1.4,(0.0+/-1.0)e+02,0.9+/-1.9,0.9+/-1.9,2,6,1.716682,1.172314,1.175492
4,CASINO,DTM,0.0025,-27.6+/-1.1,-26.8+/-1.0,(-1+/-7)e+01,-0.7+/-1.4,-0.7+/-1.5,2,6,1.242547,0.852603,0.871057
5,CHAMP-EU,LA,0.0050,-27.5+/-0.7,-25.12+/-0.19,-449+/-13,-2.4+/-0.8,-0.2+/-0.8,2,5,0.062488,0.114790,0.115489
6,CHAMP-EU,TM,0.0050,-27.9+/-0.4,-26.8+/-0.5,-275+/-33,-1.1+/-0.6,0.3+/-0.6,2,5,1.214677,0.276903,0.287634
7,CHAMP-US,LA,0.0050,-23.4+/-2.2,-25.8+/-3.1,(-0.5+/-1.5)e+02,2+/-4,3+/-4,2,5,2.741829,1.830172,1.643701
8,CHAMP-US,TM,0.0050,-26.8+/-1.9,-29.0+/-1.9,(1.2+/-0.9)e+02,2.2+/-2.7,1.6+/-2.7,2,5,1.595242,1.174550,1.005445
9,CMQMC,LA,0.0025,-25.7+/-2.0,-25.7+/-1.8,(-0.7+/-1.3)e+02,0.0+/-2.7,0.2+/-2.7,2,6,2.032470,1.496348,1.266588


In [12]:
len(df_Eint)

30

In [13]:
df_Eint.loc[ df_Eint.RCS >2 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
7,CHAMP-US,LA,0.0050,-23.4+/-2.2,-25.8+/-3.1,(-0.5+/-1.5)e+02,2+/-4,3+/-4,2,5,2.741829,1.830172,1.643701
9,CMQMC,LA,0.0025,-25.7+/-2.0,-25.7+/-1.8,(-0.7+/-1.3)e+02,0.0+/-2.7,0.2+/-2.7,2,6,2.032470,1.496348,1.266588
11,CMQMC,DLA,0.0025,-29.2+/-1.0,-27.9+/-1.2,(-8+/-9)e+01,-1.3+/-1.5,-1.1+/-1.6,2,6,2.692394,1.093787,1.088553
16,QMC=Chem,DLA,0.0025,-24.4+/-2.0,-27.4+/-3.1,(3+/-4)e+02,3+/-4,2+/-4,3,6,2.173896,1.503239,1.166897
25,QWalk,DLA,0.0025,-27.7+/-1.1,-29.5+/-1.5,(-0.3+/-1.1)e+02,1.8+/-1.9,1.8+/-1.9,2,6,3.096937,1.368005,1.322083


In [14]:
df_Eint.loc[ df_Eint.RMSR >2 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
14,PyQMC,TM,0.0025,-30+/-4,-23.2+/-1.9,(-1.1+/-0.9)e+02,-7+/-4,-6+/-4,2,7,0.930909,2.615229,1.240986


In [15]:
df_Eint.loc[ df_Eint.Delta.apply(lambda x: abs(x.nominal_value) ) >3 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
14,PyQMC,TM,0.0025,-30+/-4,-23.2+/-1.9,(-1.1+/-0.9)e+02,-7+/-4,-6+/-4,2,7,0.930909,2.615229,1.240986
16,QMC=Chem,DLA,0.0025,-24.4+/-2.0,-27.4+/-3.1,(3+/-4)e+02,3+/-4,2+/-4,3,6,2.173896,1.503239,1.166897
17,QMCPACK,LA,0.0050,-20.9+/-1.3,-17.5+/-1.6,(-1.02+/-0.11)e+03,-3.4+/-2.1,1.7+/-2.2,2,5,1.494123,0.982130,0.978357


### Etot methane

In [16]:
df = dfs_Etot['m']
df

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
0,Amolqc,DLA,0.0025,-8.084455+/-0.000030,-8.08450+/-0.00004,0.019+/-0.005,(5+/-5)e-05,(0+/-5)e-05,2,5,1.305621,0.000022,0.000021
1,CASINO,LA,0.0050,-8.081906+/-0.000018,-8.081941+/-0.000009,0.0085+/-0.0006,(3.5+/-2.0)e-05,(-0.7+/-2.0)e-05,2,5,0.247963,0.000006,0.000006
2,CASINO,TM,0.0025,-8.081012+/-0.000016,-8.081013+/-0.000020,0.0103+/-0.0016,(0.0+/-2.6)e-05,(-2.5+/-2.6)e-05,2,6,2.334522,0.000018,0.000018
3,CASINO,DLA,0.0025,-8.084444+/-0.000019,-8.084445+/-0.000017,0.0090+/-0.0014,(0.1+/-2.6)e-05,(-2.2+/-2.6)e-05,2,6,1.459715,0.000015,0.000016
4,CASINO,DTM,0.0025,-8.078509+/-0.000015,-8.078561+/-0.000009,0.0255+/-0.0008,(5.1+/-1.8)e-05,(-1.2+/-1.8)e-05,2,6,0.581418,0.000009,0.000009
5,CHAMP-EU,LA,0.0050,-8.082368+/-0.000012,-8.082432+/-0.000023,0.0168+/-0.0016,(6.4+/-2.6)e-05,(-1.9+/-2.7)e-05,2,5,3.627083,0.000014,0.000014
6,CHAMP-EU,TM,0.0050,-8.081069+/-0.000006,-8.081077+/-0.000008,0.0033+/-0.0006,(0.8+/-1.1)e-05,(-0.8+/-1.1)e-05,2,5,1.635880,0.000005,0.000005
7,CHAMP-US,LA,0.0050,-8.082519+/-0.000035,-8.08263+/-0.00009,0.018+/-0.010,0.00011+/-0.00010,0.00002+/-0.00011,3,5,2.882206,0.000028,0.000016
8,CHAMP-US,TM,0.0050,-8.081074+/-0.000029,-8.081031+/-0.000033,-0.0004+/-0.0015,(-4+/-4)e-05,(-4+/-4)e-05,2,5,1.968423,0.000023,0.000015
9,CMQMC,LA,0.0025,-8.08295+/-0.00005,-8.08309+/-0.00006,0.083+/-0.008,0.00013+/-0.00008,(-7+/-8)e-05,3,6,2.597943,0.000031,0.000022


In [17]:
df.keys()

Index(['Code', 'Method', 'tau_min', 'Etaumin', 'Etau0', 'dE/dtau', 'Delta',
       'DDelta', 'degree', 'Ndata', 'RCS', 'RMSR', 'WRMSR'],
      dtype='object')

In [18]:
print( np.mean(df.RMSR), np.std( df.RMSR ) )
df.loc[ df.RMSR > 0.0001 ]

2.0688652002856767e-05 1.1871455459735896e-05


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR


In [19]:
Delta = df.Delta.apply(lambda x: abs(x.n) ).to_numpy()
print( np.mean(Delta), np.std( Delta ) )
df.loc[ Delta > 0.0001 ]

6.0251814107085506e-05 6.561979876904889e-05


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
7,CHAMP-US,LA,0.0050,-8.082519+/-0.000035,-8.08263+/-0.00009,0.018+/-0.010,0.00011+/-0.00010,0.00002+/-0.00011,3,5,2.882206,0.000028,0.000016
9,CMQMC,LA,0.0025,-8.08295+/-0.00005,-8.08309+/-0.00006,0.083+/-0.008,0.00013+/-0.00008,(-7+/-8)e-05,3,6,2.597943,0.000031,0.000022
11,CMQMC,DLA,0.0025,-8.084233+/-0.000020,-8.08438+/-0.00005,0.070+/-0.008,0.00015+/-0.00005,(-3+/-6)e-05,3,6,4.363439,0.000024,0.000024
16,QMC=Chem,DLA,0.0025,-8.084388+/-0.000023,-8.084515+/-0.000012,0.0530+/-0.0017,0.000127+/-0.000025,(-0.5+/-2.6)e-05,3,6,0.282857,0.000006,0.000004
17,QMCPACK,LA,0.0050,-8.083062+/-0.000023,-8.08340+/-0.00008,0.077+/-0.011,0.00033+/-0.00008,-0.00005+/-0.00010,3,5,3.502084,0.000019,0.000018
21,QMeCha,LA,0.0100,-8.082482+/-0.000017,-8.08264+/-0.00009,0.021+/-0.007,0.00015+/-0.00009,-0.00005+/-0.00012,3,5,5.071098,0.000018,0.000017


In [20]:
df.loc[ df.RCS > 10 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR


### Etot water

In [21]:
df = dfs_Etot['w']
df

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
0,Amolqc,DLA,0.0025,-17.246210+/-0.000030,-17.246378+/-0.000023,0.0638+/-0.0031,0.00017+/-0.00004,(1+/-4)e-05,2,5,0.518289,0.000014,0.000014
1,CASINO,LA,0.0050,-17.240909+/-0.000021,-17.24104+/-0.00004,0.033+/-0.006,0.00013+/-0.00005,(-4+/-5)e-05,3,5,1.057873,0.000011,0.000011
2,CASINO,TM,0.0025,-17.239442+/-0.000021,-17.23950+/-0.00004,0.038+/-0.008,(6+/-5)e-05,(-4+/-5)e-05,3,6,3.380360,0.000023,0.000023
3,CASINO,DLA,0.0025,-17.246238+/-0.000026,-17.246337+/-0.000018,0.0483+/-0.0029,(9.9+/-3.1)e-05,(-2.2+/-3.2)e-05,3,6,0.394556,0.000009,0.000009
4,CASINO,DTM,0.0005,-17.23475+/-0.00007,-17.234727+/-0.000016,0.0866+/-0.0027,(-2+/-7)e-05,(-6+/-7)e-05,3,8,0.455806,0.000024,0.000012
5,CHAMP-EU,LA,0.0050,-17.241363+/-0.000013,-17.241562+/-0.000011,0.0456+/-0.0015,0.000199+/-0.000017,(-2.8+/-1.9)e-05,3,5,0.185981,0.000003,0.000003
6,CHAMP-EU,TM,0.0050,-17.239626+/-0.000008,-17.239670+/-0.000014,0.0113+/-0.0020,(4.4+/-1.6)e-05,(-1.3+/-1.9)e-05,3,5,0.996912,0.000003,0.000004
7,CHAMP-US,LA,0.0050,-17.24146+/-0.00004,-17.241671+/-0.000014,0.0485+/-0.0016,0.00021+/-0.00004,(-3+/-5)e-05,3,5,0.049753,0.000004,0.000003
8,CHAMP-US,TM,0.0050,-17.23963+/-0.00004,-17.23956+/-0.00004,-0.0080+/-0.0019,(-7+/-6)e-05,(-3+/-6)e-05,2,5,1.986411,0.000026,0.000023
9,CMQMC,LA,0.0025,-17.241772+/-0.000020,-17.24206+/-0.00010,0.153+/-0.018,0.00029+/-0.00011,-0.00010+/-0.00011,3,6,19.159328,0.000053,0.000055


In [22]:
print( np.mean(df.RMSR), np.std( df.RMSR ) )
df.loc[ df.RMSR > 0.0003 ]

2.0538226567551662e-05 1.3140739255825781e-05


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR


In [23]:
Delta = df.Delta.apply(lambda x: abs(x.n) ).to_numpy()
print( np.mean(Delta), np.std( Delta ) )
df.loc[ Delta > 0.0003 ]

0.00014927753880963717 0.00010546833815752008


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
11,CMQMC,DLA,0.0025,-17.245548+/-0.000020,-17.24588+/-0.00007,0.161+/-0.012,0.00034+/-0.00008,(-6+/-8)e-05,3,6,10.392025,0.000037,0.000037
16,QMC=Chem,DLA,0.0025,-17.245744+/-0.000023,-17.24605+/-0.00007,0.159+/-0.010,0.00030+/-0.00008,(-9+/-8)e-05,3,6,9.144790,0.000036,0.000027
20,QMCPACK,DTM,0.0025,-17.235202+/-0.000021,-17.234817+/-0.000027,-0.151+/-0.005,-0.000385+/-0.000034,(-1+/-4)e-05,3,6,1.067109,0.000014,0.000015


In [24]:
df.loc[ df.RCS >10 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
9,CMQMC,LA,0.0025,-17.241772+/-0.000020,-17.24206+/-0.00010,0.153+/-0.018,0.00029+/-0.00011,-0.00010+/-0.00011,3,6,19.159328,0.000053,0.000055
11,CMQMC,DLA,0.0025,-17.245548+/-0.000020,-17.24588+/-0.00007,0.161+/-0.012,0.00034+/-0.00008,(-6+/-8)e-05,3,6,10.392025,0.000037,0.000037


### Etot methane-water

In [25]:
df = dfs_Etot['mw']
df

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
0,Amolqc,DLA,0.0025,-25.331758+/-0.000030,-25.331904+/-0.000028,0.062+/-0.004,0.00015+/-0.00004,(-1+/-4)e-05,2,5,0.799620,0.000017,0.000017
1,CASINO,LA,0.0050,-25.324040+/-0.000026,-25.32427+/-0.00004,0.052+/-0.006,0.00023+/-0.00005,(-3+/-6)e-05,3,5,0.722336,0.000011,0.000011
2,CASINO,TM,0.0050,-25.321244+/-0.000029,-25.32149+/-0.00004,0.057+/-0.006,0.00024+/-0.00005,(-4+/-6)e-05,3,5,0.527924,0.000010,0.000010
3,CASINO,DLA,0.0025,-25.33179+/-0.00004,-25.33189+/-0.00005,0.048+/-0.008,(9+/-7)e-05,(-3+/-7)e-05,3,6,1.470340,0.000028,0.000025
4,CASINO,DTM,0.0025,-25.314040+/-0.000029,-25.314318+/-0.000034,0.121+/-0.006,0.00028+/-0.00005,(-3+/-5)e-05,3,6,1.017456,0.000017,0.000017
5,CHAMP-EU,LA,0.0050,-25.324743+/-0.000021,-25.324972+/-0.000029,0.055+/-0.004,0.00023+/-0.00004,(-4+/-4)e-05,3,5,0.615309,0.000007,0.000007
6,CHAMP-EU,TM,0.0050,-25.321721+/-0.000011,-25.321740+/-0.000014,0.0057+/-0.0020,(1.9+/-1.8)e-05,(-0.9+/-2.1)e-05,3,5,0.527877,0.000003,0.000004
7,CHAMP-US,LA,0.0050,-25.32484+/-0.00006,-25.32501+/-0.00006,0.034+/-0.006,0.00017+/-0.00008,(-0+/-9)e-05,3,5,0.424864,0.000018,0.000013
8,CHAMP-US,TM,0.0050,-25.32169+/-0.00005,-25.321654+/-0.000017,-0.0040+/-0.0008,(-4+/-5)e-05,(-2+/-5)e-05,2,5,0.172781,0.000013,0.000009
9,CMQMC,LA,0.0025,-25.32567+/-0.00006,-25.32611+/-0.00013,0.238+/-0.016,0.00044+/-0.00014,-0.00016+/-0.00015,3,6,6.596281,0.000058,0.000039


In [26]:
print( np.mean(df.RMSR), np.std( df.RMSR ) )
df.loc[ df.RMSR > 0.0003 ]

2.639166575768948e-05 2.213672462811726e-05


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR


In [27]:
Delta = df.Delta.apply(lambda x: abs(x.n) ).to_numpy()
print( np.mean(Delta), np.std( Delta ) )
df.loc[ Delta > 0.0003 ]

0.0001909446049026542 0.00014932018416899883


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
9,CMQMC,LA,0.0025,-25.32567+/-0.00006,-25.32611+/-0.00013,0.238+/-0.016,0.00044+/-0.00014,-0.00016+/-0.00015,3,6,6.596281,0.000058,0.000039
11,CMQMC,DLA,0.0025,-25.330854+/-0.000021,-25.33128+/-0.00014,0.227+/-0.024,0.00043+/-0.00015,-0.00014+/-0.00016,3,6,37.626181,0.000072,0.000071
12,CMQMC,DTM,0.0025,-25.314434+/-0.000020,-25.314108+/-0.000035,-0.120+/-0.006,-0.00033+/-0.00004,(-3+/-4)e-05,3,6,2.274788,0.000017,0.000017
16,QMC=Chem,DLA,0.0025,-25.33103+/-0.00007,-25.331574+/-0.000033,0.225+/-0.005,0.00055+/-0.00007,(-2+/-7)e-05,3,6,0.224659,0.000016,0.000013
17,QMCPACK,LA,0.0050,-25.325507+/-0.000035,-25.32592+/-0.00005,0.096+/-0.006,0.00041+/-0.00006,(-7+/-7)e-05,3,5,0.477295,0.000010,0.000010
18,QMCPACK,TM,0.0025,-25.322093+/-0.000028,-25.321764+/-0.000016,-0.1377+/-0.0027,-0.000328+/-0.000032,(1.6+/-3.3)e-05,3,6,0.214625,0.000008,0.000009
20,QMCPACK,DTM,0.0025,-25.31493+/-0.00004,-25.31443+/-0.00007,-0.208+/-0.012,-0.00050+/-0.00008,(2+/-9)e-05,3,6,3.127505,0.000036,0.000037


In [28]:
df.loc[ df.RCS >10 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
11,CMQMC,DLA,0.0025,-25.330854+/-0.000021,-25.33128+/-0.00014,0.227+/-0.024,0.00043+/-0.00015,-0.00014+/-0.00016,3,6,37.626181,0.000072,0.000071


### Etot methane-far-water

In [29]:
df = dfs_Etot['m---w']
df

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
0,Amolqc,DLA,0.0025,-25.330740+/-0.000030,-25.330842+/-0.000004,0.0495+/-0.0008,0.000103+/-0.000030,(-2.1+/-3.0)e-05,2,4,0.006573,0.000001,0.000001
1,CASINO,LA,0.0050,-25.322904+/-0.000026,-25.323055+/-0.000005,0.0368+/-0.0007,0.000150+/-0.000027,(-3.4+/-2.7)e-05,3,5,0.010226,0.000001,0.000001
2,CASINO,TM,0.0050,-25.320190+/-0.000025,-25.320468+/-0.000032,0.064+/-0.005,0.00028+/-0.00004,(-4+/-5)e-05,3,5,0.444613,0.000008,0.000009
3,CASINO,DLA,0.0025,-25.330781+/-0.000033,-25.330879+/-0.000034,0.053+/-0.006,0.00010+/-0.00005,(-4+/-5)e-05,3,6,0.786049,0.000017,0.000017
4,CASINO,DTM,0.0025,-25.313010+/-0.000030,-25.313278+/-0.000023,0.119+/-0.004,0.00027+/-0.00004,(-3+/-4)e-05,3,6,0.442866,0.000011,0.000011
5,CHAMP-EU,LA,0.0050,-25.323963+/-0.000024,-25.324239+/-0.000020,0.0625+/-0.0026,0.000277+/-0.000031,(-3.6+/-3.3)e-05,3,5,0.238191,0.000004,0.000004
6,CHAMP-EU,TM,0.0050,-25.320719+/-0.000010,-25.320718+/-0.000033,0.004+/-0.005,(-0.1+/-3.4)e-05,(-2+/-4)e-05,3,5,3.038405,0.000008,0.000008
7,CHAMP-US,LA,0.0050,-25.32399+/-0.00005,-25.324219+/-0.000023,0.0502+/-0.0027,0.00023+/-0.00006,(-2+/-6)e-05,3,5,0.081420,0.000007,0.000005
8,CHAMP-US,TM,0.0050,-25.32068+/-0.00004,-25.32059+/-0.00007,-0.008+/-0.004,(-8+/-8)e-05,(-4+/-8)e-05,2,5,2.856283,0.000040,0.000038
9,CMQMC,LA,0.0025,-25.32476+/-0.00009,-25.32501+/-0.00023,0.223+/-0.025,0.00025+/-0.00024,-0.00031+/-0.00025,3,6,10.407623,0.000125,0.000049


In [30]:
print( np.mean(df.RMSR), np.std( df.RMSR ) )
df.loc[ df.RMSR > 0.0003 ]

3.191944004298485e-05 4.013831224311797e-05


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR


In [31]:
Delta = df.Delta.apply(lambda x: abs(x.n) ).to_numpy()
print( np.mean(Delta), np.std( Delta ) )
df.loc[ Delta > 0.0003 ]

0.00019104969607243542 0.0001425930448049225


,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
11,CMQMC,DLA,0.0025,-25.329793+/-0.000020,-25.33028+/-0.00012,0.235+/-0.020,0.00049+/-0.00012,-0.00010+/-0.00013,3,6,26.497310,0.000059,0.000059
12,CMQMC,DTM,0.0025,-25.313462+/-0.000020,-25.31312+/-0.00007,-0.117+/-0.011,-0.00034+/-0.00007,(-5+/-8)e-05,3,6,8.520129,0.000034,0.000034
16,QMC=Chem,DLA,0.0025,-25.33005+/-0.00006,-25.330546+/-0.000023,0.2175+/-0.0033,0.00049+/-0.00006,(-5+/-6)e-05,3,6,0.149406,0.000011,0.000009
17,QMCPACK,LA,0.0050,-25.324615+/-0.000032,-25.32502+/-0.00016,0.100+/-0.023,0.00041+/-0.00017,-0.00009+/-0.00020,3,5,7.406086,0.000041,0.000041
20,QMCPACK,DTM,0.0025,-25.313873+/-0.000033,-25.31336+/-0.00007,-0.219+/-0.012,-0.00051+/-0.00008,(4+/-8)e-05,3,6,3.413731,0.000038,0.000038


In [32]:
df.loc[ df.RCS >10 ]

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
9,CMQMC,LA,0.0025,-25.32476+/-0.00009,-25.32501+/-0.00023,0.223+/-0.025,0.00025+/-0.00024,-0.00031+/-0.00025,3,6,10.407623,0.000125,0.000049
11,CMQMC,DLA,0.0025,-25.329793+/-0.000020,-25.33028+/-0.00012,0.235+/-0.020,0.00049+/-0.00012,-0.00010+/-0.00013,3,6,26.497310,0.000059,0.000059


## Make table Eint fit & extrapolations

In [33]:
df_Eint.keys()

Index(['Code', 'Method', 'tau_min', 'Etaumin', 'Etau0', 'dE/dtau', 'Delta',
       'DDelta', 'degree', 'Ndata', 'RCS', 'RMSR', 'WRMSR'],
      dtype='object')

In [34]:
Table_Eint_column_formats = {
    'Code'     : '{:<16}',
    'Method'   : '{:^5}',
    'tau_min'  : '{:.4f}',
    'Etaumin'  : '{:.1f}',
    'Etau0'    : '{:.1f}',
    'Delta'    : '{:.1f}',
    'degree'   : '{:}',
    'Ndata'    : '{:}',
    'RCS'      : '{:.1f}',
    'RMSR'     : '{:.1f}',
    #'WRMSR'    : '{:.1f}',
}

replacements = {
    '+/-'      : '$\pm$',
    'Etaumin'  : r'$E_\text{int}^{\tau_\text{min}}$',
    'tau_min'  : r'$\tau_\text{min}$',
    'Etau0'    : r'$E_\text{int}^{\tau\to 0}$',
    'Delta'    : r'$\Delta E_\text{int}^{\tau_\text{min},0}$',
    'degree'   : r'$d_\text{poly}$',
    'Ndata'    : r'$N$',
    'RCS'      : r'$\chi^2_\textrm{red}$',
}

Table_Eint = df_Eint.loc[:,Table_Eint_column_formats.keys()].style.format(Table_Eint_column_formats).hide(axis='index').to_latex(
    hrules=True, column_format='|lc|lr|r|r|cccc|')

for old, new in replacements.items():
    Table_Eint = Table_Eint.replace( old, new )

#print( Table_Etot )
with open('Tables/Table_Eint.tex','w') as file:
    file.write( Table_Eint )

## Make table Etot fit & extrapolations

In [35]:
dfs_Etot['m'].keys()

Index(['Code', 'Method', 'tau_min', 'Etaumin', 'Etau0', 'dE/dtau', 'Delta',
       'DDelta', 'degree', 'Ndata', 'RCS', 'RMSR', 'WRMSR'],
      dtype='object')

In [36]:
len(_)

13

In [37]:
Table_Etot_column_formats = {
    'Code'     : '{:<16}',
    'Method'   : '{:^5}',
    'tau_min'  : '{:.4f}',
    'Etaumin'  : '{:.5fS}',
    'Etau0'    : '{:.5fS}',
    'Delta'    : '{:.5fS}',
    'degree'   : '{:}',
    'Ndata'    : '{:}',
    'RCS'      : '{:.1f}',
    'RMSR'     : '{:.5f}',
    #'WRMSR'    : '{:.1f}',
}

replacements = {
    '+/-'      : '$\pm$',
    'Etaumin'  : r'$E_\text{tot}^{\tau_\text{min}}$',
    'tau_min'  : r'$\tau_\text{min}$',
    'Etau0'    : r'$E_\text{tot}^{\tau\to 0}$',
    'Delta'    : r'$\Delta E_\text{tot}^{\tau_\text{min},0}$',
    'degree'   : r'$d_\text{poly}$',
    'Ndata'    : r'$N$',
    'RCS'      : r'$\chi^2_\textrm{red}$',
}

In [38]:
for system in systems:
    
    Table_Etot = dfs_Etot[system].loc[:,Table_Etot_column_formats.keys()].style.format(Table_Etot_column_formats).hide(axis='index').to_latex(
        hrules=True, column_format='|lc|lr|r|r|cccc|')
    
    for old, new in replacements.items():
        Table_Etot = Table_Etot.replace( old, new )
    
    #print( Table_Etot )
    with open(f'Tables/Table_Etot_{system}.tex','w') as file:
        file.write( Table_Etot )

# Average estimations Eint

In [39]:
for method in methods:
    print( method, method_codes_dict[method])
    print( mean_codes(final_interaction_energy,method_codes_dict[method], method) )

LA ['CASINO', 'CHAMP-EU', 'CHAMP-US', 'CMQMC', 'PyQMC', 'QMCPACK', 'QMeCha', 'QWalk']
(-27.4548944809344, 7.212612142864175)
TM ['CASINO', 'CHAMP-EU', 'CHAMP-US', 'CMQMC', 'PyQMC', 'QMCPACK', 'QWalk', 'TurboRVB-DMC', 'TurboRVB-LRDMC']
(-27.094930080853743, 2.3994587879472653)
DLA ['Amolqc', 'CASINO', 'CMQMC', 'PyQMC', 'QMC=Chem', 'QMCPACK', 'QMeCha', 'QWalk', 'TurboRVB-LRDMC']
(-29.055824371225626, 1.9436681157039717)
DLTM ['CASINO', 'CMQMC', 'QMCPACK', 'TurboRVB-LRDMC']
(-27.92696326907891, 1.0901234467908534)


In [40]:
d = {}

for method in ['LA','TM','DLA','DTM']:
    ene_i = df_Eint.loc[ df_Eint.Method==method ].Etau0.apply( lambda x: x.n ).to_numpy()
    err_i = df_Eint.loc[ df_Eint.Method==method ].Etau0.apply( lambda x: x.s ).to_numpy()
    
    mean_ene = ene_i.mean()
    var_ene = np.mean( err_i**2 ) + np.mean( (ene_i - mean_ene)**2 )
    err_ene = np.sqrt( var_ene )

    d[method] = ufloat( mean_ene, err_ene )
    print(f'{method} Ene = {mean_ene} +/- {err_ene}')

print(d)
mean_Eint = pd.Series( d, name='Eint[meV]' )
mean_Eint

LA Ene = -27.4548944809344 +/- 7.212612142864184
TM Ene = -27.094930080853747 +/- 2.399458787947261
DLA Ene = -29.055824371225626 +/- 1.943668115703936
DTM Ene = -27.92696326907891 +/- 1.090123446790934
{'LA': -27.4548944809344+/-7.212612142864184, 'TM': -27.094930080853747+/-2.399458787947261, 'DLA': -29.055824371225626+/-1.943668115703936, 'DTM': -27.92696326907891+/-1.090123446790934}


LA         -27+/-7
TM     -27.1+/-2.4
DLA    -29.1+/-1.9
DTM    -27.9+/-1.1
Name: Eint[meV], dtype: object

In [41]:
# table_mean_Eint = mean_Eint.reset_index()
# table_mean_Eint.columns = ['','Interaction Energy']
# table_mean_Eint.to_csv('Tables/df_mean_Eint.csv',index=False)
# table_mean_Eint

# Table_mean_formats = {
#     table_mean_Eint.columns[-1]  : '{:.1f}',
# }

# replacements = {
#     '+/-'      : '$\pm$',
# }

# Latextable_mean_Eint = table_mean_Eint.style.format(Table_mean_formats).hide(axis='index').to_latex(
#     hrules=True, column_format='|l c|')

# for old, new in replacements.items():
#     Latextable_mean_Eint = Latextable_mean_Eint.replace( old, new )

# with open('Tables/Table_mean_Eint.tex','w') as file:
#     file.write( Latextable_mean_Eint )

## Agreement

In [42]:
mean_Eint.keys()

Index(['LA', 'TM', 'DLA', 'DTM'], dtype='object')

In [43]:
d = {}
for method in mean_Eint.keys():
    ref = mean_Eint.loc[method].n
    res = df_Eint.loc[ df_Eint.Method == method ].Etau0.to_numpy()
    res_nv = np.array([ x.nominal_value for x in res ])
    diffs = res_nv - ref
    print(f'Method {method}: max-absolute-deviation={max(abs(diffs)):.1f} meanAD={np.mean(abs(diffs)):.1f}') 
    print(f'   max difference among predictions of 2 codes={max(diffs)-min(diffs):.1f}')
    print(f'Codes: {df_Eint.loc[ df_Eint.Method == method ].Code.to_numpy()}')
    print( res-ref )
    print( 60*'-')
    d[method] = [ mean_Eint[method], len(diffs), max(diffs)-min(diffs), max(abs(diffs)), np.mean(abs(diffs)), np.sqrt(np.mean( diffs**2)) ]

df_mean_Eint = pd.DataFrame( d ).T
df_mean_Eint.columns = [ 'Mean','No. codes','Max-dist','Max-error','MAE','RMSE' ]
df_mean_Eint.to_csv('Tables/df_mean_Eint.csv',index=False)
df_mean_Eint

Method LA: max-absolute-deviation=11.3 meanAD=6.1
   max difference among predictions of 2 codes=21.3
Codes: ['CASINO' 'CHAMP-EU' 'CHAMP-US' 'CMQMC' 'PyQMC' 'QMCPACK' 'QMeCha' 'QWalk']
[-5.31606683875728+/-1.2136930950152724
 2.33469977004415+/-0.19123627874887797
 1.6320300851925431+/-3.085430053070111
 1.710225048656966+/-1.7597610694620467
 8.576074685986722+/-0.9205677814250905
 9.974905781164363+/-1.6264879488471158
 -7.591297163318636+/-0.28879270612633046
 -11.320571368968821+/-0.5838745790318475]
------------------------------------------------------------
Method TM: max-absolute-deviation=3.9 meanAD=1.5
   max difference among predictions of 2 codes=6.4
Codes: ['CASINO' 'CHAMP-EU' 'CHAMP-US' 'CMQMC' 'PyQMC' 'QMCPACK' 'QWalk'
 'TurboRVB-DMC' 'TurboRVB-LRDMC']
[-0.127544558655039+/-0.9495701913765818
 0.29134656945781146+/-0.46103883392273026
 -1.897058571466367+/-1.9395676465033163
 -1.0996121295201675+/-0.8858186850074847
 3.8900839082689664+/-1.9168047495507055
 2.62722378745

,Mean,No. codes,Max-dist,Max-error,MAE,RMSE
LA,-27+/-7,8,21.295477,11.320571,6.056984,7.054526
TM,-27.1+/-2.4,9,6.422802,3.890084,1.535634,1.970343
DLA,-29.1+/-1.9,9,3.798962,2.151312,1.074823,1.229969
DTM,-27.9+/-1.1,4,2.132588,1.07769,0.601883,0.76031


In [44]:
Table_mean_formats = { i : '{:.1f}' for i in df_mean_Eint.columns }
Table_mean_formats['No. codes'] = '{}'
    

replacements = {
    '+/-'      : '$\pm$',
}

Latextable_mean_Eint = df_mean_Eint.style.format(Table_mean_formats).to_latex(
    hrules=True, column_format='|l c c c c c c|')

for old, new in replacements.items():
    Latextable_mean_Eint = Latextable_mean_Eint.replace( old, new )

with open('Tables/Table_mean_Eint.tex','w') as file:
    file.write( Latextable_mean_Eint )

# Average estimation of Etot

In [45]:
for system in ['m','w','mw']:
    for method in methods:
        print( system, method, method_codes_dict[method])
        print( mean_codes_tot_ene(final_total_energy,method_codes_dict[method], method, system) )

#mean_codes(final_total_energy,method_codes_dict[method],method)

m LA ['CASINO', 'CHAMP-EU', 'CHAMP-US', 'CMQMC', 'PyQMC', 'QMCPACK', 'QMeCha', 'QWalk']
(-8.082643316910891, 0.0004961403772541546)
m TM ['CASINO', 'CHAMP-EU', 'CHAMP-US', 'CMQMC', 'PyQMC', 'QMCPACK', 'QWalk', 'TurboRVB-DMC', 'TurboRVB-LRDMC']
(-8.081000726721577, 8.298971211365002e-05)
m DLA ['Amolqc', 'CASINO', 'CMQMC', 'PyQMC', 'QMC=Chem', 'QMCPACK', 'QMeCha', 'QWalk', 'TurboRVB-LRDMC']
(-8.084446329390945, 6.5271546396735e-05)
m DLTM ['CASINO', 'CMQMC', 'QMCPACK', 'TurboRVB-LRDMC']
(-8.078553749477729, 4.585573244192411e-05)
w LA ['CASINO', 'CHAMP-EU', 'CHAMP-US', 'CMQMC', 'PyQMC', 'QMCPACK', 'QMeCha', 'QWalk']
(-17.24157037500066, 0.00042329081055957985)
w TM ['CASINO', 'CHAMP-EU', 'CHAMP-US', 'CMQMC', 'PyQMC', 'QMCPACK', 'QWalk', 'TurboRVB-DMC', 'TurboRVB-LRDMC']
(-17.239550228551444, 0.00011971794328640303)
w DLA ['Amolqc', 'CASINO', 'CMQMC', 'PyQMC', 'QMC=Chem', 'QMCPACK', 'QMeCha', 'QWalk', 'TurboRVB-LRDMC']
(-17.246189370046675, 0.00017399024955363753)
w DLTM ['CASINO', 'CMQM

In [46]:
dfs_Etot['m']

,Code,Method,tau_min,Etaumin,Etau0,dE/dtau,Delta,DDelta,degree,Ndata,RCS,RMSR,WRMSR
0,Amolqc,DLA,0.0025,-8.084455+/-0.000030,-8.08450+/-0.00004,0.019+/-0.005,(5+/-5)e-05,(0+/-5)e-05,2,5,1.305621,0.000022,0.000021
1,CASINO,LA,0.0050,-8.081906+/-0.000018,-8.081941+/-0.000009,0.0085+/-0.0006,(3.5+/-2.0)e-05,(-0.7+/-2.0)e-05,2,5,0.247963,0.000006,0.000006
2,CASINO,TM,0.0025,-8.081012+/-0.000016,-8.081013+/-0.000020,0.0103+/-0.0016,(0.0+/-2.6)e-05,(-2.5+/-2.6)e-05,2,6,2.334522,0.000018,0.000018
3,CASINO,DLA,0.0025,-8.084444+/-0.000019,-8.084445+/-0.000017,0.0090+/-0.0014,(0.1+/-2.6)e-05,(-2.2+/-2.6)e-05,2,6,1.459715,0.000015,0.000016
4,CASINO,DTM,0.0025,-8.078509+/-0.000015,-8.078561+/-0.000009,0.0255+/-0.0008,(5.1+/-1.8)e-05,(-1.2+/-1.8)e-05,2,6,0.581418,0.000009,0.000009
5,CHAMP-EU,LA,0.0050,-8.082368+/-0.000012,-8.082432+/-0.000023,0.0168+/-0.0016,(6.4+/-2.6)e-05,(-1.9+/-2.7)e-05,2,5,3.627083,0.000014,0.000014
6,CHAMP-EU,TM,0.0050,-8.081069+/-0.000006,-8.081077+/-0.000008,0.0033+/-0.0006,(0.8+/-1.1)e-05,(-0.8+/-1.1)e-05,2,5,1.635880,0.000005,0.000005
7,CHAMP-US,LA,0.0050,-8.082519+/-0.000035,-8.08263+/-0.00009,0.018+/-0.010,0.00011+/-0.00010,0.00002+/-0.00011,3,5,2.882206,0.000028,0.000016
8,CHAMP-US,TM,0.0050,-8.081074+/-0.000029,-8.081031+/-0.000033,-0.0004+/-0.0015,(-4+/-4)e-05,(-4+/-4)e-05,2,5,1.968423,0.000023,0.000015
9,CMQMC,LA,0.0025,-8.08295+/-0.00005,-8.08309+/-0.00006,0.083+/-0.008,0.00013+/-0.00008,(-7+/-8)e-05,3,6,2.597943,0.000031,0.000022


In [47]:
l = []
for system in systems:
    d = {}
    for method in ['LA','TM','DLA','DTM']:
        ene_i = dfs_Etot[system].loc[ dfs_Etot[system].Method==method ].Etau0.apply( lambda x: x.n ).to_numpy()
        err_i = dfs_Etot[system].loc[ dfs_Etot[system].Method==method ].Etau0.apply( lambda x: x.s ).to_numpy()
        
        mean_ene = ene_i.mean()
        var_ene = np.mean( err_i**2 ) + np.mean( (ene_i - mean_ene)**2 )
        err_ene = np.sqrt( var_ene )
        d[method] = ufloat( mean_ene, err_ene )
        
        print(f'{system} {method} Ene = {mean_ene} +/- {err_ene}')
    l.append(d)
mean_Etot = pd.DataFrame(l).T
mean_Etot.columns = systems

mean_Etot.to_csv( 'Tables/df_mean_Etot.csv' )

mean_Etot

m LA Ene = -8.082643316910893 +/- 0.0004961403600286645
m TM Ene = -8.081000726721577 +/- 8.298972880847492e-05
m DLA Ene = -8.084446329390945 +/- 6.527159764654942e-05
m DTM Ene = -8.078553749477729 +/- 4.5855560719425495e-05
w LA Ene = -17.241570375000663 +/- 0.0004232907862609455
w TM Ene = -17.239550228551444 +/- 0.00011971766018161605
w DLA Ene = -17.246189370046675 +/- 0.00017399043234278227
w DTM Ene = -17.234732922448043 +/- 8.769967006102473e-05
mw LA Ene = -25.325206550360765 +/- 0.0006771855900644453
mw TM Ene = -25.321549829292824 +/- 0.00015363239954642945
mw DLA Ene = -25.331706899232 +/- 0.0002310781447474257
mw DTM Ene = -25.314326262498675 +/- 0.0001424005040580789
m---w LA Ene = -25.324243647683332 +/- 0.0007773968690503477
m---w TM Ene = -25.320465565091617 +/- 0.00025702348580524336
m---w DLA Ene = -25.33065003071894 +/- 0.000244373672686208
m---w DTM Ene = -25.313284699217846 +/- 0.00011527438060245593


,m,w,mw,m---w
LA,-8.0826+/-0.0005,-17.2416+/-0.0004,-25.3252+/-0.0007,-25.3242+/-0.0008
TM,-8.08100+/-0.00008,-17.23955+/-0.00012,-25.32155+/-0.00015,-25.32047+/-0.00026
DLA,-8.08445+/-0.00007,-17.24619+/-0.00017,-25.33171+/-0.00023,-25.33065+/-0.00024
DTM,-8.07855+/-0.00005,-17.23473+/-0.00009,-25.31433+/-0.00014,-25.31328+/-0.00012


In [48]:
mean_Etot.loc[:,['m','w','mw']]

,m,w,mw
LA,-8.0826+/-0.0005,-17.2416+/-0.0004,-25.3252+/-0.0007
TM,-8.08100+/-0.00008,-17.23955+/-0.00012,-25.32155+/-0.00015
DLA,-8.08445+/-0.00007,-17.24619+/-0.00017,-25.33171+/-0.00023
DTM,-8.07855+/-0.00005,-17.23473+/-0.00009,-25.31433+/-0.00014


In [49]:
mean_Etot.m.apply(lambda x: f'{x.n} +/- {x.s}' )

LA      -8.082643316910893 +/- 0.0004961403600286645
TM      -8.081000726721577 +/- 8.298972880847492e-05
DLA     -8.084446329390945 +/- 6.527159764654942e-05
DTM    -8.078553749477729 +/- 4.5855560719425495e-05
Name: m, dtype: object

In [50]:
mean_Etot.m.apply(lambda x: f'{x:.5f}' )

LA     -8.08264+/-0.00050
TM     -8.08100+/-0.00008
DLA    -8.08445+/-0.00007
DTM    -8.07855+/-0.00005
Name: m, dtype: object

In [51]:
l1 = mean_Etot.columns
rep_index = {
    'm' : 'Methane',
    'w' : 'Water',
    'mw': 'Methane-Water',
}
l2 = [ rep_index[x] for x in l1 if x in rep_index.keys()  ]
l2
mean_Etot_n = mean_Etot.loc[:,rep_index.keys()]
mean_Etot_n.columns = l2
mean_Etot_n

,Methane,Water,Methane-Water
LA,-8.0826+/-0.0005,-17.2416+/-0.0004,-25.3252+/-0.0007
TM,-8.08100+/-0.00008,-17.23955+/-0.00012,-25.32155+/-0.00015
DLA,-8.08445+/-0.00007,-17.24619+/-0.00017,-25.33171+/-0.00023
DTM,-8.07855+/-0.00005,-17.23473+/-0.00009,-25.31433+/-0.00014


In [52]:
# mean_Etot.index.name = 'Method'

In [53]:
Table_mean_Etot_column_formats = {
    'Methane'      : '{:.5f}',
    'Water'        : '{:.5f}',
    'Methane-Water': '{:.5f}',
}

replacements = {
    '+/-'      : '$\pm$',
}

Table_mean_Etot = mean_Etot_n.loc[:,Table_mean_Etot_column_formats.keys()].style.format(Table_mean_Etot_column_formats).to_latex(
    hrules=True, column_format='|l c c c|')

for old, new in replacements.items():
    Table_mean_Etot = Table_mean_Etot.replace( old, new )

with open('Tables/Table_mean_Etot.tex','w') as file:
    file.write( Table_mean_Etot )

In [54]:
# Etot mean prop.

d = {}
for system in systems:
    for method in mean_Etot[system].keys():
        ref = mean_Etot.loc[method,system].n
        res = dfs_Etot[system].loc[ dfs_Etot[system].Method == method ].Etau0.to_numpy()
        res_nv = np.array([ x.nominal_value for x in res ])
        diffs = res_nv - ref
        print(f'Method {method}: max-absolute-deviation={max(abs(diffs)):.1f} meanAD={np.mean(abs(diffs)):.1f}') 
        print(f'   max difference among predictions of 2 codes={max(diffs)-min(diffs):.1f}')
        print(f'Codes: {dfs_Etot[system].loc[ dfs_Etot[system].Method == method ].Code.to_numpy()}')
        print( res-ref )
        print( 60*'-')
        d[f'{system} {method}'] = [ mean_Etot.loc[method,system], len(diffs), max(diffs)-min(diffs), max(abs(diffs)), np.mean(abs(diffs)), np.sqrt(np.mean( diffs**2)) ]

df_meanp_Etot = pd.DataFrame( d ).T
df_meanp_Etot.columns = [ 'Mean','No. codes','Max-dist','Max-error','MAE','RMSE' ]
df_meanp_Etot.to_csv('Tables/df_meanp_Etot.csv',index=False)
df_meanp_Etot

Method LA: max-absolute-deviation=0.0 meanAD=0.0
   max difference among predictions of 2 codes=0.0
Codes: ['CASINO' 'CHAMP-EU' 'CHAMP-US' 'CMQMC' 'PyQMC' 'QMCPACK' 'QMeCha' 'QWalk']
[0.0007026012385491498+/-9.368752048850937e-06
 0.00021095888494215842+/-2.306062451996642e-05
 1.6340756893384878e-05+/-9.334767106005556e-05
 -0.0004439737150434553+/-5.986743650051714e-05
 -0.000421582941186216+/-3.0229175666397913e-05
 -0.0007519984848709527+/-7.829116297196203e-05
 7.801399261708752e-06+/-8.675164434795697e-05
 0.0006798528614613275+/-2.6998704155082063e-05]
------------------------------------------------------------
Method TM: max-absolute-deviation=0.0 meanAD=0.0
   max difference among predictions of 2 codes=0.0
Codes: ['CASINO' 'CHAMP-EU' 'CHAMP-US' 'CMQMC' 'PyQMC' 'QMCPACK' 'QWalk'
 'TurboRVB-DMC' 'TurboRVB-LRDMC']
[-1.2048013973142702e-05+/-2.0314644040980647e-05
 -7.66237071285758e-05+/-8.475442749953265e-06
 -3.018258836817722e-05+/-3.320765749111644e-05
 9.859762960218177e-0

,Mean,No. codes,Max-dist,Max-error,MAE,RMSE
m LA,-8.0826+/-0.0005,8,0.001455,0.000752,0.000404,0.000493
m TM,-8.08100+/-0.00008,9,0.000253,0.000154,0.000061,0.000075
m DLA,-8.08445+/-0.00007,9,0.000172,0.000104,0.00005,0.000057
m DTM,-8.07855+/-0.00005,4,0.000113,0.000071,0.000036,0.000043
w LA,-17.2416+/-0.0004,8,0.001191,0.000702,0.000357,0.00042
w TM,-17.23955+/-0.00012,9,0.000376,0.000217,0.000096,0.000114
w DLA,-17.24619+/-0.00017,9,0.000493,0.000305,0.000149,0.000164
w DTM,-17.23473+/-0.00009,4,0.000222,0.000138,0.000072,0.000086
mw LA,-25.3252+/-0.0007,8,0.001857,0.000957,0.000583,0.000673
mw TM,-25.32155+/-0.00015,9,0.000457,0.000242,0.000116,0.000145


## Figure 1 - Comparison among different codes

In [55]:
fig , ax = plt.subplots(4,1,figsize=(6.67,8),dpi=600,constrained_layout=True,sharex=True,sharey=True)

counter = 0
for method in ['LA','TM','DLA','DLTM']:
    ax[counter].axhline(y=final_ccsdtq_ref,color='black',ls='--')

    for code in method_codes_dict[method]:
        ax[counter].errorbar(code_to_num_dict[code],final_interaction_energy[code][method][0],final_interaction_energy[code][method][1],fmt='o',capsize=6,ms=10,elinewidth=1.5,markeredgewidth=1,mec='black',c=dmc_code_info[code]['color'])

    ax[counter].axhline(mean_codes(final_interaction_energy,method_codes_dict[method],method)[0],c='grey',ls='dashed',lw=2)
    ax[counter].fill_between(np.arange(-0.5,13.5,0.1),mean_codes(final_interaction_energy,method_codes_dict[method],method)[0] + mean_codes(final_interaction_energy,method_codes_dict[method],method)[1],mean_codes(final_interaction_energy,method_codes_dict[method],method)[0] - mean_codes(final_interaction_energy,method_codes_dict[method],method)[1] ,color='grey',alpha=.3,edgecolor='none')
    ax[counter].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
    ax[counter].set_xticklabels([])
    ax[counter].set_xlim([-0.5,11.5])
    ax[counter].set_ylim([-40,-15])
    ax[counter].set_ylabel('Interaction energy [meV]')

    counter += 1

ax[0].set_title('Locality Approximation (LA)')
ax[1].set_title('T-Move (TM)')
ax[2].set_title('Determinant Locality Approximation (DLA)')
ax[3].set_title('Determinant Locality Approximation with T-Move (DTM)',fontsize=11)

ax[3].set_xticklabels([dmc_code_info[x]['label'] for x in code_to_num_dict],rotation=90) 

plt.savefig('Figures/Fig_01.png')

## Figure 2 - Distribution of errors for different methods

In [56]:
# fig , ax = plt.subplots(figsize=(3.37,4),dpi=600,constrained_layout=True)

# x=np.arange(-55,-10,0.1)
# heightstep = 0.6

# ax.axvline(final_ccsdtq_ref,ls='dashed',lw=2,c='black')

# # Plot distribution for LA
# y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['LA'],'LA',dist_info=True)
# ax.errorbar(mean,heightstep*3 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='blue')
# ax.plot(x,y_dist+heightstep*3,label='LA', lw=2,c='blue')
# ax.fill_between(x,y_dist+heightstep*3,heightstep*3,color='blue',alpha=0.2)
# ax.axhline(heightstep*3,ls='dashed',lw=0.5,c='blue')
# t = ax.text(-52,heightstep*3 + 0.1,'LA',fontsize=15)
# t.set_bbox(dict(facecolor='blue', alpha=0.5, edgecolor='blue'))
# ax.text(-45,heightstep*3 + 0.17,r'$\mu_\textrm{LA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
# ax.text(-45,heightstep*3 + 0.07,r'$\sigma_\textrm{LA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# # Plot distribution for TM
# y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['TM'],'TM',dist_info=True)
# ax.errorbar(mean,heightstep*2 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='orange')
# ax.plot(x,y_dist+heightstep*2,label='TM', lw=2,c='orange')
# ax.fill_between(x,y_dist+heightstep*2,heightstep*2,color='orange',alpha=0.2)
# ax.axhline(heightstep*2,ls='dashed',lw=0.5,c='orange')
# t = ax.text(-52,heightstep * 2 + 0.1,'TM',fontsize=15)
# t.set_bbox(dict(facecolor='orange', alpha=0.5, edgecolor='orange'))
# ax.text(-45,heightstep*2 + 0.17,r'$\mu_\textrm{TM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
# ax.text(-45,heightstep*2 + 0.07,r'$\sigma_\textrm{TM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# # Plot distribution for DLA
# y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLA'],'DLA',dist_info=True)
# ax.errorbar(mean,heightstep - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='red')
# ax.plot(x,y_dist+heightstep,label='DLA', lw=2,c='red')
# ax.fill_between(x,y_dist+heightstep,heightstep,color='red',alpha=0.2)
# ax.axhline(heightstep,ls='dashed',lw=0.5,c='red')
# t = ax.text(-52,heightstep + 0.1,'DLA',fontsize=15)
# t.set_bbox(dict(facecolor='red', alpha=0.5, edgecolor='red'))
# ax.text(-45,heightstep + 0.17,r'$\mu_\textrm{DLA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
# ax.text(-45,heightstep + 0.07,r'$\sigma_\textrm{DLA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# # Plot distribution for DLTM
# y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLTM'],'DLTM',dist_info=True)
# ax.errorbar(mean, -0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='green')
# ax.plot(x,y_dist,label='DLTM', lw=2,c='green')
# ax.fill_between(x,y_dist,0,color='green',alpha=0.2)
# ax.axhline(0,ls='dashed',lw=0.5,c='green')
# t = ax.text(-52,0.1,'DTM',fontsize=15)
# t.set_bbox(dict(facecolor='green', alpha=0.5, edgecolor='green'))
# ax.text(-45, 0.17,r'$\mu_\textrm{DTM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
# ax.text(-45, 0.07,r'$\sigma_\textrm{DTM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# ax.spines[['right', 'top','left']].set_visible(False)

# plt.ylim([-0.2,3.25])
# plt.yticks([])
# plt.xlim([-55,-15])

# plt.xlabel('Interaction energy [meV]')
# plt.savefig('Figures/Fig_02.png')


In [212]:
fig , ax = plt.subplots(figsize=(3.37,4),dpi=600,constrained_layout=True)

x=np.arange(-55,-10,0.1)
heightstep = [ 0., 1., 1.6, 2.2 ]

shift_method = 0.2
shift_mu = 0.27
shift_sigma = 0.17

ax.axvline(final_ccsdtq_ref,ls='dashed',lw=2,c='black')

# Plot distribution for LA
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['LA'],'LA',dist_info=True)
ax.errorbar(mean,heightstep[3] - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='blue')
ax.plot(x,y_dist+heightstep[3],label='LA', lw=2,c='blue')
ax.fill_between(x,y_dist+heightstep[3],heightstep[3],color='blue',alpha=0.2)
ax.axhline(heightstep[3],ls='dashed',lw=0.5,c='blue')
t = ax.text(-52,heightstep[3] + shift_method,'LA',fontsize=15)
t.set_bbox(dict(facecolor='blue', alpha=0.5, edgecolor='none'))
ax.text(-45,heightstep[3] + shift_mu,r'$\mu_\textrm{LA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep[3] + shift_sigma,r'$\sigma_\textrm{LA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for TM
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['TM'],'TM',dist_info=True)
ax.errorbar(mean,heightstep[2] - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='orange')
ax.plot(x,y_dist+heightstep[2],label='TM', lw=2,c='orange')
ax.fill_between(x,y_dist+heightstep[2],heightstep[2],color='orange',alpha=0.2)
ax.axhline(heightstep[2],ls='dashed',lw=0.5,c='orange')
t = ax.text(-52,heightstep[2] + shift_method,'TM',fontsize=15)
t.set_bbox(dict(facecolor='orange', alpha=0.5, edgecolor='none'))
ax.text(-45,heightstep[2] + shift_mu,r'$\mu_\textrm{TM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep[2] + shift_sigma,r'$\sigma_\textrm{TM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for DLA
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLA'],'DLA',dist_info=True)
ax.errorbar(mean,heightstep[1] - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='red')
ax.plot(x,y_dist+heightstep[1],label='DLA', lw=2,c='red')
ax.fill_between(x,y_dist+heightstep[1],heightstep[1],color='red',alpha=0.2)
ax.axhline(heightstep[1],ls='dashed',lw=0.5,c='red')
t = ax.text(-52,heightstep[1] + shift_method,'DLA',fontsize=15)
t.set_bbox(dict(facecolor='red', alpha=0.5, edgecolor='none'))
ax.text(-45,heightstep[1] + shift_mu,r'$\mu_\textrm{DLA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep[1] + shift_sigma,r'$\sigma_\textrm{DLA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for DLTM
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLTM'],'DLTM',dist_info=True)
ax.errorbar(mean, -0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='green')
ax.plot(x,y_dist,label='DLTM', lw=2,c='green')
ax.fill_between(x,y_dist,0,color='green',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='green')
t = ax.text(-52,shift_method,'DTM',fontsize=15)
t.set_bbox(dict(facecolor='green', alpha=0.5, edgecolor='none'))
ax.text(-45, shift_mu,r'$\mu_\textrm{DTM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45, shift_sigma,r'$\sigma_\textrm{DTM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


ax.spines[['right', 'top','left']].set_visible(False)

plt.ylim([-0.2,2.6])
plt.yticks([])
plt.xlim([-55,-15])

plt.xlabel('Interaction energy [meV]')
plt.savefig('Figures/Fig_02.png')


## Figure 3 - Timestep dependence of distribution of codes

In [58]:
fig , ax = plt.subplots(figsize=(3.37,5),dpi=600)

x=np.arange(-85,-10,0.1)
offset=0.5

ax.axvline(final_ccsdtq_ref,ls='dashed',lw=2,c='black')

# Get the interaction energies for the codes which have the same set of timesteps
code_list_tau_DLA=['Amolqc','CASINO','CMQMC','PyQMC','QMC=Chem','QMCPACK']
DLA_interaction_energy = {code: dmc_code_data[code]['interaction_energy']['raw']['DLA'] for code in code_list_tau_DLA}


# Plot data for 0.0400 timestep
y_dist, mean, error = gauss_tau(x,DLA_interaction_energy,code_list_tau_DLA,0.0400)
ax.errorbar(mean,offset*5 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='blue')
ax.plot(x,y_dist+offset*5,label=r'$\tau$ = 0.04 a.u.',lw=2,c='blue')
ax.fill_between(x,y_dist+offset*5,offset*5,color='blue',alpha=0.2)
ax.axhline(offset*5,ls='dashed',lw=0.5,c='blue')
t = ax.text(-75,0.12+offset*5,r'$\tau$ = 0.04 a.u.',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='blue'))
ax.text(-48,offset*5 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*5 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot data for 0.0200 timestep
y_dist, mean, error = gauss_tau(x,DLA_interaction_energy,code_list_tau_DLA,0.0200)
ax.errorbar(mean,offset*4 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='dodgerblue')
ax.plot(x,y_dist+offset*4,label=r'$\tau$ = 0.02 a.u.',lw=2,c='dodgerblue')
ax.fill_between(x,y_dist+offset*4,offset*4,color='dodgerblue',alpha=0.2)
ax.axhline(offset*4,ls='dashed',lw=0.5,c='dodgerblue')
t = ax.text(-75,0.12+offset*4,r'$\tau$ = 0.02 a.u.',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='dodgerblue'))
ax.text(-48,offset*4 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*4 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for 0.0100 timestep
y_dist, mean, error = gauss_tau(x,DLA_interaction_energy,code_list_tau_DLA,0.0100)
ax.errorbar(mean,offset*3 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='turquoise')
ax.plot(x,y_dist+offset*3,label=r'$\tau$ = 0.01 a.u.',lw=2,c='turquoise')
ax.fill_between(x,y_dist+offset*3,offset*3,color='turquoise',alpha=0.2)
ax.axhline(offset*3,ls='dashed',lw=0.5,c='turquoise')
t = ax.text(-75,0.12+offset*3,r'$\tau$ = 0.01 a.u.',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='turquoise'))
ax.text(-48,offset*3 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*3 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for 0.0050 timestep
y_dist, mean, error = gauss_tau(x,DLA_interaction_energy,code_list_tau_DLA,0.0050)
ax.errorbar(mean,offset*2 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='orange')
ax.plot(x,y_dist+offset*2,label=r'$\tau$ = 0.005 a.u.',lw=2,c='orange')
ax.fill_between(x,y_dist+offset*2,offset*2,color='orange',alpha=0.2)
ax.axhline(offset*2,ls='dashed',lw=0.5,c='orange')
t = ax.text(-75,0.12+offset*2,r'$\tau$ = 0.005 a.u.',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='orange'))
ax.text(-48,offset*2 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*2 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for 0.0025 timestep
y_dist, mean, error = gauss_tau(x,DLA_interaction_energy,code_list_tau_DLA,0.0025)
ax.errorbar(mean,offset - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='firebrick')
ax.plot(x,y_dist+offset,label=r'$\tau$ = 0.0025 a.u.', lw=2,c='firebrick')
ax.fill_between(x,y_dist+offset,offset,color='firebrick',alpha=0.2)
ax.axhline(offset,ls='dashed',lw=0.5,c='firebrick')
t = ax.text(-75,0.12+offset,r'$\tau$ = 0.0025 a.u.',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='firebrick'))
ax.text(-48,offset + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for timestep -> 0
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLA'],'DLA',dist_info=True)
ax.errorbar(mean,- 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='red')
ax.plot(x,y_dist,label=r'$\tau$ $\to$ 0 a.u.', lw=2,c='red')
ax.fill_between(x,y_dist,0,color='red',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='red')
t = ax.text(-75,0.12,r'$\tau$ $\to$ 0 a.u.',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='red'))
ax.text(-48, + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48, + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


ax.spines[['right', 'top','left']].set_visible(False)

plt.yticks([])
plt.xlim([-85,-10])
plt.xlabel('Interaction energy [meV]')

plt.tight_layout()
plt.savefig('Figures/Fig_03.png')

/home/shixubenjamin/Projects/DMC/Analysis/DMC-reproducibility/Scripts/myfunctions.py:171: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  gauss += stats.norm.pdf(x, a[i].loc[tau][0], a[i].loc[tau][1])
/home/shixubenjamin/Projects/DMC/Analysis/DMC-reproducibility/Scripts/myfunctions.py:172: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  energy += a[i].loc[tau][0]
/home/shixubenjamin/Projects/DMC/Analysis/DMC-reproducibility/Scripts/myfunctions.py:180: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame be

## SI - Time-step dependence of interaction energy of individual codes

In [59]:
fig, axs = plt.subplots(2,4, figsize=(6.69,4.5), sharey=True, sharex=True,dpi=600,constrained_layout=True)

# Plot the top row interaction energies
counter = 0
for counter, method in enumerate(['LA','TM','DLA','DLTM']):
    for code in ['CASINO', 'CMQMC','QMCPACK', 'TurboRVB-DMC','TurboRVB-LRDMC']:
        if code in method_codes_dict[method]:
            fitting_data = dmc_code_data[code]['interaction_energy']['fit'][method]
            axs[0,counter].errorbar(fitting_data['x'], fitting_data['E'],yerr = fitting_data['E_err'], fmt='o', markersize=3,elinewidth=1, label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])
            axs[0,counter].plot( fitting_data['x_fit'], fitting_data['E_fit'], '--', color=dmc_code_info[code]['color'], alpha = 0.6 )
            axs[0,counter].fill_between(fitting_data['x_fit'],fitting_data['E_fit_lower_lim'],fitting_data['E_fit_upper_lim'],color=dmc_code_info[code]['color'],alpha=0.2)
    axs[0,counter].axhline(y=final_ccsdtq_ref,color='black',ls='--',label='CCSD(T)')


# Plot the top row interaction energies
for counter, method in enumerate(['LA','TM','DLA']):
    for code in ['Amolqc','CHAMP-EU','CHAMP-US','PyQMC','QMC=Chem','QMeCha','QWalk']:
        if code in method_codes_dict[method]:
            fitting_data = dmc_code_data[code]['interaction_energy']['fit'][method]
            axs[1,counter].errorbar(fitting_data['x'], fitting_data['E'],yerr = fitting_data['E_err'], fmt='o', markersize=3,elinewidth=1, label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])
            axs[1,counter].plot( fitting_data['x_fit'], fitting_data['E_fit'], '--', color=dmc_code_info[code]['color'], alpha = 0.6 )
            axs[1,counter].fill_between(fitting_data['x_fit'],fitting_data['E_fit_lower_lim'],fitting_data['E_fit_upper_lim'],color=dmc_code_info[code]['color'],alpha=0.2)
    axs[1,counter].axhline(y=final_ccsdtq_ref,color='black',ls='--',label='CCSD(T)')

# Create the legend
for code in dmc_code_info:
    axs[1,3].bar([1],[5], label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])

axs[1,3].axis('off')
axs[1,3].legend(fancybox=False, fontsize=7)

axs[0,0].set_title('LA')
axs[0,1].set_title('TM')
axs[0,2].set_title('DLA')
axs[0,3].set_title('DTM')

axs[0,0].set_xlim([0.0,0.09])
axs[0,0].set_ylim([-70,-20])
axs[0,0].set_ylabel('Interaction energy [meV]')
axs[1,0].set_ylabel('Interaction energy [meV]')

axs[1,0].set_xticks([0.00,0.03,0.06,0.09])
axs[1,0].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,1].set_xticks([0.00,0.03,0.06,0.09])
axs[1,1].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,2].set_xticks([0.00,0.03,0.06,0.09])
axs[1,2].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,1].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,2].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )

plt.savefig('Figures/Fig_SI_interaction_energy_timestep_dependence.png')

## SI - Time-step dependence of total energies of individual codes

In [60]:
fig, axs = plt.subplots(3,4, figsize=(6.69,7),  sharey=True, sharex=True,dpi=600,constrained_layout=True)


for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
    counter = 0
    for indexj, method in enumerate(['LA','TM','DLA','DLTM']):
        mean_ene, mean_ene_std = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,tot_ene_sys_label)
        for code in method_codes_dict[method]:
            fitting_data = dmc_code_data[code]['total_energy']['fit'][f'{tot_ene_sys_label}_{method}']

            axs[indexsys,indexj].errorbar(fitting_data['x'], fitting_data['E'] - mean_ene,yerr = fitting_data['E_err'], fmt='o', markersize=3,elinewidth=1, color=dmc_code_info[code]['color'])
            axs[indexsys,indexj].plot( fitting_data['x_fit'], fitting_data['E_fit'] - mean_ene, '--', linewidth=1, color=dmc_code_info[code]['color'], alpha = 0.6 )
            axs[indexsys,indexj].fill_between(fitting_data['x_fit'],fitting_data['E_fit_lower_lim'] - mean_ene,fitting_data['E_fit_upper_lim'] - mean_ene,color=dmc_code_info[code]['color'],alpha=0.2)

        axs[indexsys,indexj].axhline(0.0,c='grey',ls='dashed',lw=1, label=f'Avg={mean_ene:6.4f}(' f'{int(round(mean_ene_std*10000))})')
        axs[indexsys,indexj].legend(loc='upper right',fontsize=8)

        axs[indexsys,indexj].set_ylim([- 0.002, + 0.002])


for code in dmc_code_info:
    axs[0,3].bar([1],[5],label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])


axs[0,0].set_title('LA')
axs[0,1].set_title('TM')
axs[0,2].set_title('DLA')
axs[0,3].set_title('DTM')

axs[0,0].set_xlim([0.0,0.09])
axs[0,0].set_ylabel('Methane--Water')
axs[1,0].set_ylabel('Methane')
axs[2,0].set_ylabel('Water ')
fig.supylabel('Deviation from average total energy [Ha]')


axs[2,0].set_xticks([0.00,0.03,0.06,0.09])
axs[2,0].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,1].set_xticks([0.00,0.03,0.06,0.09])
axs[2,1].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,2].set_xticks([0.00,0.03,0.06,0.09])
axs[2,2].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,3].set_xticks([0.00,0.03,0.06,0.09])
axs[2,3].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[2,1].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[2,2].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[2,3].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )

plt.savefig('Figures/Fig_SI_Total_energy_timestep_dependence.png')

In [211]:
from matplotlib.ticker import AutoMinorLocator


final_system_total_energy = {
    'mw': {code: {method: final_total_energy[code][method]['mw'] for method in final_total_energy[code]} for code in final_total_energy},
    'm': {code: {method: final_total_energy[code][method]['m'] for method in final_total_energy[code]} for code in final_total_energy},
    'w': {code: {method: final_total_energy[code][method]['w'] for method in final_total_energy[code]} for code in final_total_energy}
}

final_system_total_energy_mean_and_dist = {system: {method: {} for method in ['LA','TM','DLA','DLTM']} for system in ['mw','m','w']}

for system in ['mw','m','w']:
    for method in ['LA','TM','DLA','DLTM']:
        final_system_total_energy_mean_and_dist[system][method]['mean'] = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=system)[0]
        final_system_total_energy_mean_and_dist[system][method]['std'] = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=system)[1]
        x=np.arange(round(final_system_total_energy_mean_and_dist[system][method]['mean'],4) - 0.004,round(final_system_total_energy_mean_and_dist[system][method]['mean'],4)+0.004,0.00005)[:160]
        y_dist, _,_ = gauss(x,final_system_total_energy[system],method_codes_dict[method],method,dist_info=True)
        final_system_total_energy_mean_and_dist[system][method]['x'] = x
        final_system_total_energy_mean_and_dist[system][method]['y_dist'] = y_dist

fig , ax = plt.subplots(1,3,figsize=(6.659,4),dpi=600,constrained_layout=True)

heightstep = [ 0., 14000, 24000, 34000 ]

shift_method = 4000
shift_mu = 4800
shift_sigma = 3300

shift_x_mean = 0.00025

x = np.arange(-0.004,0.004,0.00005)[:160]

for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
    ax[indexsys].axvline(0.0,ls='dashed',lw=1,c='black',alpha=0.3)



    method = 'LA'
    ax[indexsys].errorbar(0,heightstep[3] - 1300,xerr=final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['std'],fmt='o',capsize=3,ms=0,elinewidth=1.5,markeredgewidth=1.5,c='blue', markeredgecolor='none')
    ax[indexsys].plot(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist']+heightstep[3],label='LA', lw=1,c='blue')
    ax[indexsys].fill_between(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist']+heightstep[3],heightstep[3],color='blue',alpha=0.2)
    ax[indexsys].axhline(heightstep[3],ls='dashed',lw=0.5,c='blue')
    if indexsys == 0:
        t = ax[indexsys].text(- 0.001,heightstep[3] + shift_method,'LA',fontsize=15)
        t.set_bbox(dict(facecolor='blue', alpha=0.5, edgecolor='none'))
    ax[indexsys].text(shift_x_mean,heightstep[3] + shift_mu,r'$\mu_\textrm{LA} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean + 0.00085,heightstep[3] + shift_mu,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["mean"]:7.5f}',fontsize=8, ha='right')
    ax[indexsys].text(shift_x_mean,heightstep[3] + shift_sigma,r'$\sigma_\textrm{LA} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean + 0.00085,heightstep[3] + shift_sigma,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["std"]:7.5f}',fontsize=8, ha='right')


    method = 'TM'
    ax[indexsys].errorbar(0,heightstep[2] - 1300,xerr=final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['std'],fmt='o',capsize=3,ms=0,elinewidth=1.5,markeredgewidth=1.5,c='orange',markeredgecolor='none')
    ax[indexsys].plot(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist']+heightstep[2],label='TM', lw=1,c='orange')
    ax[indexsys].fill_between(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist']+heightstep[2],heightstep[2],color='orange',alpha=0.2)
    ax[indexsys].axhline(heightstep[2],ls='dashed',lw=0.5,c='orange')
    if indexsys == 0:
        t = ax[indexsys].text(- 0.001,heightstep[2] + shift_method,'TM',fontsize=15)
        t.set_bbox(dict(facecolor='orange', alpha=0.5, edgecolor='none'))
    ax[indexsys].text(shift_x_mean,heightstep[2] + shift_mu,r'$\mu_\textrm{TM} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean + 0.00085,heightstep[2] + shift_mu,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["mean"]:7.5f}',fontsize=8, ha='right')
    ax[indexsys].text(shift_x_mean,heightstep[2] + shift_sigma,r'$\sigma_\textrm{TM} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean + 0.00085,heightstep[2] + shift_sigma,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["std"]:7.5f}',fontsize=8, ha='right')

    method = 'DLA'
    ax[indexsys].errorbar(0,heightstep[1] - 1300,xerr=final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['std'],fmt='o',capsize=3,ms=0,elinewidth=1.5,markeredgewidth=1.5,c='red', markeredgecolor='none')
    ax[indexsys].plot(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist']+heightstep[1],label='DLA', lw=1,c='red')
    ax[indexsys].fill_between(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist']+heightstep[1],heightstep[1],color='red',alpha=0.2)
    ax[indexsys].axhline(heightstep[1],ls='dashed',lw=0.5,c='red')
    if indexsys == 0:
        t = ax[indexsys].text(- 0.001,heightstep[1] + shift_method,'DLA',fontsize=15)
        t.set_bbox(dict(facecolor='red', alpha=0.5, edgecolor='none'))
    ax[indexsys].text(shift_x_mean,heightstep[1] + shift_mu,r'$\mu_\textrm{DLA} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean + 0.00085,heightstep[1] + shift_mu,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["mean"]:7.5f}',fontsize=8, ha='right')
    ax[indexsys].text(shift_x_mean,heightstep[1] + shift_sigma,r'$\sigma_\textrm{DLA} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean + 0.00085,heightstep[1] + shift_sigma,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["std"]:7.5f}',fontsize=8, ha='right')

    method = 'DLTM'
    ax[indexsys].errorbar(0, - 1300,xerr=final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['std'],fmt='o',capsize=3,ms=0,elinewidth=1.5,markeredgewidth=1.5,c='green', markeredgecolor='none')
    ax[indexsys].plot(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist'],label='DLTM', lw=1,c='green')
    ax[indexsys].fill_between(x,final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]['y_dist'],0,color='green',alpha=0.2)
    ax[indexsys].axhline(0,ls='dashed',lw=0.5,c='green')
    if indexsys == 0:
        t = ax[indexsys].text(- 0.001,shift_method,'DTM',fontsize=15)
        t.set_bbox(dict(facecolor='green', alpha=0.5, edgecolor='none'))
    ax[indexsys].text(shift_x_mean, shift_mu,r'$\mu_\textrm{DTM} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean+0.00085, shift_mu,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["mean"]:7.5f}',fontsize=8,ha='right')
    ax[indexsys].text(shift_x_mean, shift_sigma,r'$\sigma_\textrm{DTM} \, {=}\,$',fontsize=8)
    ax[indexsys].text(shift_x_mean+0.00085, shift_sigma,f'{final_system_total_energy_mean_and_dist[tot_ene_sys_label][method]["std"]:7.5f}',fontsize=8,ha='right')    


    ax[indexsys].spines[['right', 'top','left']].set_visible(False)
    ax[indexsys].set_xlim([-0.0012,0.0012])
    ax[indexsys].set_ylim([-3000,41000])
    ax[indexsys].set_yticks([])
    ax[indexsys].set_xlabel('Deviation from mean [Ha]')
    ax[indexsys].set_xticks([-0.001,0.0,0.001])
    # Set minor ticks
    ax[indexsys].xaxis.set_minor_locator(AutoMinorLocator(5))

ax[0].set_title('Methane-Water')
ax[1].set_title('Methane')
ax[2].set_title('Water')

# plt.xlabel('Interaction energy [meV]')
plt.savefig('Figures/Fig_03_new.png')

In [ ]:
final_system_total_energy = {
    'mw': {code: {method: final_total_energy[code][method]['mw'] for method in final_total_energy[code]} for code in final_total_energy},
    'm': {code: {method: final_total_energy[code][method]['m'] for method in final_total_energy[code]} for code in final_total_energy},
    'w': {code: {method: final_total_energy[code][method]['w'] for method in final_total_energy[code]} for code in final_total_energy}
}

fig , ax = plt.subplots(1,3,figsize=(6.659,4),dpi=600,constrained_layout=True)

x=np.arange(-55,-10,0.1)
heightstep = [ 0., 1., 1.6, 2.2 ]

shift_method = 0.2
shift_mu = 0.27
shift_sigma = 0.17

# ax.axvline(final_ccsdtq_ref,ls='dashed',lw=2,c='black')

# Plot distribution for LA
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['LA'],'LA',dist_info=True)
ax.errorbar(mean,heightstep[3] - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='blue')
ax.plot(x,y_dist+heightstep[3],label='LA', lw=2,c='blue')
ax.fill_between(x,y_dist+heightstep[3],heightstep[3],color='blue',alpha=0.2)
ax.axhline(heightstep[3],ls='dashed',lw=0.5,c='blue')
t = ax.text(-52,heightstep[3] + shift_method,'LA',fontsize=15)
t.set_bbox(dict(facecolor='blue', alpha=0.5, edgecolor='blue'))
ax.text(-45,heightstep[3] + shift_mu,r'$\mu_\textrm{LA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep[3] + shift_sigma,r'$\sigma_\textrm{LA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for TM
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['TM'],'TM',dist_info=True)
ax.errorbar(mean,heightstep[2] - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='orange')
ax.plot(x,y_dist+heightstep[2],label='TM', lw=2,c='orange')
ax.fill_between(x,y_dist+heightstep[2],heightstep[2],color='orange',alpha=0.2)
ax.axhline(heightstep[2],ls='dashed',lw=0.5,c='orange')
t = ax.text(-52,heightstep[2] + shift_method,'TM',fontsize=15)
t.set_bbox(dict(facecolor='orange', alpha=0.5, edgecolor='orange'))
ax.text(-45,heightstep[2] + shift_mu,r'$\mu_\textrm{TM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep[2] + shift_sigma,r'$\sigma_\textrm{TM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for DLA
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLA'],'DLA',dist_info=True)
ax.errorbar(mean,heightstep[1] - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='red')
ax.plot(x,y_dist+heightstep[1],label='DLA', lw=2,c='red')
ax.fill_between(x,y_dist+heightstep[1],heightstep[1],color='red',alpha=0.2)
ax.axhline(heightstep[1],ls='dashed',lw=0.5,c='red')
t = ax.text(-52,heightstep[1] + shift_method,'DLA',fontsize=15)
t.set_bbox(dict(facecolor='red', alpha=0.5, edgecolor='red'))
ax.text(-45,heightstep[1] + shift_mu,r'$\mu_\textrm{DLA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep[1] + shift_sigma,r'$\sigma_\textrm{DLA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for DLTM
y_dist, mean, error = gauss(x,final_interaction_energy,method_codes_dict['DLTM'],'DLTM',dist_info=True)
ax.errorbar(mean, -0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='green')
ax.plot(x,y_dist,label='DLTM', lw=2,c='green')
ax.fill_between(x,y_dist,0,color='green',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='green')
t = ax.text(-52,shift_method,'DTM',fontsize=15)
t.set_bbox(dict(facecolor='green', alpha=0.5, edgecolor='green'))
ax.text(-45, shift_mu,r'$\mu_\textrm{DTM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45, shift_sigma,r'$\sigma_\textrm{DTM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


ax.spines[['right', 'top','left']].set_visible(False)

plt.ylim([-0.2,2.6])
plt.yticks([])
plt.xlim([-55,-15])

plt.xlabel('Interaction energy [meV]')
plt.savefig('Figures/Fig_02.png')


## SI - Deviation of zero time-step total energy of methane-water dimer, methane, water

In [61]:
sys_to_nice_name_dict = {
    'mw': 'Methane-Water',
    'm': 'Methane',
    'w': 'Water'
}

# figure 1 Comparison among different codes 
fig , ax = plt.subplots(4,3,figsize=(6.67,8),dpi=600,constrained_layout=True,sharex=True, sharey=True)


for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
    counter = 0
    for method in ['LA','TM','DLA','DLTM']:
        mean_ene = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=tot_ene_sys_label)[0]
        mean_ene_std = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=tot_ene_sys_label)[1]
        for code in method_codes_dict[method]:
            ax[counter,indexsys].errorbar(code_to_num_dict[code],final_total_energy[code][method][tot_ene_sys_label][0] - mean_ene,final_total_energy[code][method][tot_ene_sys_label][1],fmt='o',capsize=5,ms=6,elinewidth=1.5,markeredgewidth=1,mec='black',c=dmc_code_info[code]['color'])


        ax[counter,indexsys].axhline(0.0,c='grey',ls='dashed',lw=2, label=f'Avg={mean_ene:6.4f}(' f'{int(round(mean_ene_std*10000))})')
        ax[counter,indexsys].fill_between(np.arange(-0.5,11.5,0.1),0.0 + mean_ene_std,0.0 - mean_ene_std,color='grey',alpha=.3)
        ax[counter,indexsys].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
        ax[counter,indexsys].set_xticklabels([])
        ax[counter,indexsys].set_xlim([-0.5,11.5])
        ax[counter,indexsys].set_ylim([- 0.002, + 0.002])
        ax[counter,indexsys].legend()
        ax[counter,indexsys].set_title(f'{method_rename(method)} - {sys_to_nice_name_dict[tot_ene_sys_label]}')
        counter += 1

fig.supylabel('Deviation from average total energy [Ha]')

ax[3,0].set_xticklabels([dmc_code_info[code]['label'] for code in dmc_code_info],rotation=90) 
ax[3,1].set_xticklabels([dmc_code_info[code]['label'] for code in dmc_code_info],rotation=90) 
ax[3,2].set_xticklabels([dmc_code_info[code]['label'] for code in dmc_code_info],rotation=90) 

plt.savefig('Figures/Fig_SI_System_total_energy_deviation.png')

## SI - Total energy and interaction energy time step dependence of DMC codes

In [62]:
method_colors = {
    'LA': color_dict['red'],
    'TM': color_dict['blue'],
    'DLA': color_dict['green'],
    'DLTM': color_dict['orange']
}

for indexc, code_name in enumerate(dmc_code_info.keys()):
    fig, axs = plt.subplots(2,3,figsize=(6.69,5),constrained_layout=True, sharey='row', gridspec_kw={'height_ratios': [1,1.5]}, dpi=600)
    gs = axs[1, 2].get_gridspec()
    # remove the underlying axes
    for ax in axs[0,:]:
        ax.remove()
    axbig = fig.add_subplot(gs[0,:])
    for method in ['LA','TM','DLA','DLTM']:
        if code_name in method_codes_dict[method]:
            fitting_data = dmc_code_data[code_name]['interaction_energy']['fit'][method]

            axbig.errorbar(fitting_data['x'], 
                           fitting_data['E'],
                           yerr = fitting_data['E_err'], 
                           fmt='o', markersize=3,elinewidth=1, color=method_colors[method])
            
            label = f'{method_rename(method)} -- ' + r'$E_\textrm{int}^{\tau  \to 0} = $ ' + f'{final_interaction_energy[code_name][method][0]:3.1f}' + r'${\pm}$' + f'{final_interaction_energy[code_name][method][1]:3.1f} meV'
            if dmc_code_info[code_name]['discretization'] == 'a':
                label = label.replace('\\tau','a')
                
            axbig.plot(fitting_data['x_fit'], 
                       fitting_data['E_fit'], 
                       '--', linewidth=1, color=method_colors[method], alpha = 0.6, 
                       label=label)
            
            axbig.fill_between(fitting_data['x_fit'],
                               fitting_data['E_fit_lower_lim'],
                               fitting_data['E_fit_upper_lim'],
                               color=method_colors[method],alpha=0.2)

            for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
                final_system_energy = final_total_energy[code_name][method][tot_ene_sys_label][0]
                final_system_energy_err = final_total_energy[code_name][method][tot_ene_sys_label][1]                
                final_system_energy_u = ufloat( final_system_energy, final_system_energy_err )
                fitting_data = dmc_code_data[code_name]['total_energy']['fit'][f'{tot_ene_sys_label}_{method}']

                axs[1,indexsys].errorbar(fitting_data['x'], 
                                         fitting_data['E'] - final_system_energy,
                                         yerr = fitting_data['E_err'], 
                                         fmt='o', markersize=3,elinewidth=1, color=method_colors[method])
                
                label=r'$E_\textrm{tot}^{\tau  \to 0}=$ ' +f'{final_system_energy_u:.1uS} Ha'
                if dmc_code_info[code_name]['discretization'] == 'a':
                    label = label.replace('\\tau','a')
                
                axs[1,indexsys].plot(fitting_data['x_fit'], 
                                     fitting_data['E_fit'] - final_system_energy, 
                                     '--', linewidth=1, color=method_colors[method], alpha = 0.6, 
                                     label = label )
                
                axs[1,indexsys].fill_between(fitting_data['x_fit'],
                                             fitting_data['E_fit_lower_lim'] - final_system_energy ,
                                             fitting_data['E_fit_upper_lim'] - final_system_energy,
                                             color=method_colors[method],alpha=0.2)
                # axs[indexsys+1].axhline(0.0,c='grey',ls='dashed',lw=1)

    axbig.set_ylabel('Interaction energy [meV]')
    axbig.legend(ncol=2, loc='upper left', fontsize=8)
    axbig.set_xlim([0.0,0.09])
    axbig.set_ylim([-70,0])
    axbig.set_title(f'{dmc_code_info[code_name]["label"]}')


    for i123 in range(3):
        axs[1,i123].legend(fontsize=8, loc = 'lower left')
        axs[1,i123].set_xlim([0.0,0.09])
        axs[1,i123].set_ylim([-0.007, 0.005])

    axs[1,0].set_ylabel(r'Deviation from $E_\textrm{tot}^{\tau  \to 0}$ [Ha]')
    axs[1,0].set_title('Methane--Water')
    axs[1,1].set_title('Methane')
    axs[1,2].set_title('Water')

    if code_name == 'TurboRVB-LRDMC':
        axbig.set_xlabel( r'LRDMC $a^2$ [a.u.]' ) 
        axs[1,0].set_xlabel( r'LRDMC $a^2$ [a.u.]' )
        axs[1,1].set_xlabel( r'LRDMC $a^2$ [a.u.]' )
        axs[1,2].set_xlabel( r'LRDMC $a^2$ [a.u.]' )    
    else:
        axbig.set_xlabel( r'DMC $\tau$ [a.u.]' )
        axs[1,0].set_xlabel( r'DMC $\tau$ [a.u.]' )
        axs[1,1].set_xlabel( r'DMC $\tau$ [a.u.]' )
        axs[1,2].set_xlabel( r'DMC $\tau$ [a.u.]' )

            
    plt.savefig(f'Figures/Fig_SI_details_{code_name}.png')


In [63]:
# for indexc, code_name in enumerate(dmc_code_info.keys()):
#     available_methods = [x for x in ['LA','TM','DLA','DLTM'] if code_name in method_codes_dict[x]]
#     available_methods_str = ", ".join(available_methods)
#     nice_code_name = code_name.replace("_","-")
#     print(r"\subsection{" + nice_code_name + r"}")
#     preamble = "\\" + """begin{figure*}[!h]
#     \includegraphics[width=6.66in]{"""+ f"Figures/Fig_SI_{code_name}_details.png" + """}
#     \caption{\label{fig:""" + f"{code_name}_energy_plots" + """} The time step dependence of the (a) methane-water interaction energy (b) methane-water dimer total energy, (c) isolated methane molecule total energy, (d) isolated water molecule total energy in the""" + f" {nice_code_name} code across the {available_methods_str} algorithm(s)." + """}
# \end{figure*}
# """ + """
# \clearpage"""
#     print(preamble)

## SI - Assessing size-consistency error in interaction energy

In [64]:
method_to_subplt_dict = {
    'LA': (0,0),
    'TM': (0,1),
    'DLA': (1,0),
    'DLTM': (1,1)
}

# Plotting the difference between DLA and DLTM against LA0 and TM0 for the water molecule system

fig, axs = plt.subplots(2,2, figsize=(6.69,5), sharey=True, sharex=True, dpi=600, constrained_layout=True )

fig.supylabel('Size Consistency Error [meV]' )
fig.supxlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )

for i, method in enumerate(methods):
    axs[method_to_subplt_dict[method]].set_xlim(1e-3,0.1); axs[method_to_subplt_dict[method]].set_xscale('log')

    for code in method_codes_dict[method]:
        code_data = dmc_code_data[code]['size_consistency_error']['raw'][method]
        code_data['x'] = x_axis( code_data, dmc_code_info[code]['discretization'] )
        axs[method_to_subplt_dict[method]].errorbar( code_data.x, code_data.ene, code_data.err, fmt='o:', color=dmc_code_info[code]['color'] )

    axs[method_to_subplt_dict[method]].axhline(y=0.0,color='black',ls='--')

    axs[method_to_subplt_dict[method]].set_title(method_rename(method))
    axs[method_to_subplt_dict[method]].set_ylim([-60,20])
    axs[method_to_subplt_dict[method]].set_xlim([1e-3,0.1])


for code in dmc_code_info:
    axs[1,1].bar([2],[5],label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])

axs[1,1].legend(ncol=2, loc='lower left',fontsize=7)

fig.savefig('Figures/Fig_SI_Size_consistency_error.png')

## SI - Validation of DLA and DLTM algorithms with no Jastrow

In [65]:
# Getting data for LA0 and TM0

data_LA0 = {}

# Setting up the dictionary for methods to compare (i.e. DLA against LA0 and DLTM against TM0)
method_to_no_Jastrow = {
    'DLA': 'LA0',
    'DLTM': 'TM0'
}

no_Jastrow_available_codes = ['Amolqc','CASINO','CMQMC','QMC=Chem','QMCPACK','QMeCha','TurboRVB']

h2o_total_energy_validation = {method: {} for method in ['DLA','DLTM']}

for code in dmc_code_info: #Amolqc','CASINO','CMQMC','QMC=Chem','QMCPACK','QMeCha','TurboRVB']:
    for method in ['DLA','DLTM']:
        try:
            energy_data_no_J = pd.read_csv(
                        f'Codes_Data/{dmc_code_info[code]["methods_and_folders"][method]}/energy_w_{method_to_no_Jastrow[method]}.csv',
                        names=[dmc_code_info[code]["discretization"], 'ene','err'], decimal='.', header=None, index_col=dmc_code_info[code]["discretization"])
            energy_data_with_J = pd.read_csv(
                        f'Codes_Data/{dmc_code_info[code]["methods_and_folders"][method]}/energy_w_{method}.csv',
                        names=[dmc_code_info[code]["discretization"], 'ene','err'], decimal='.', header=None, index_col=dmc_code_info[code]["discretization"])
            df = pd.concat( { 'wJ':energy_data_with_J, 'woJ':energy_data_no_J }, axis=1 ) 
            df.sort_index(inplace=True)
            df[('DE','ene')] =  ( df[('wJ','ene')] - df[('woJ','ene')] )
            df[('DE','err')] =  ( df[('wJ','err')]**2 + df[('woJ','err')]**2 )**0.5
            h2o_total_energy_validation[method][code] = df['DE'].copy()
        except:
            pass


# Plotting the difference between DLA and DLTM against LA0 and TM0 for the water molecule system

fig, axs = plt.subplots(1,2, figsize=(6.69,4), sharey=True, sharex=True, dpi=600, constrained_layout=True )

for i, method in enumerate(['DLA','DLTM']):
    axs[i].axhline(y=0.0,color='black',ls='--',label='')
    axs[i].set_xlim(0,0.025)
    axs[i].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
    for code in h2o_total_energy_validation[method]:
        axs[i].errorbar( 
            x_axis( h2o_total_energy_validation[method][code], dmc_code_info[code]['discretization'] ), 
            h2o_total_energy_validation[method][code].ene, 
            h2o_total_energy_validation[method][code].err, 
            fmt='o:', color=dmc_code_info[code]['color'] )


axs[0].set_title('DLA')
axs[1].set_title('DTM')
axs[0].set_xlim((0.,0.021))
axs[0].set_ylim((-0.01,0.01))
axs[0].set_ylabel(r'Energy diff. between DLA (DTM) and LA0 (TM0) for H$_2$O [Ha]' )

for code in set([x for x in h2o_total_energy_validation['DLA']] + [x for x in h2o_total_energy_validation['DLTM']]):
    axs[1].bar([2],[5],label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])

axs[1].legend(ncol=2, loc='upper left',fontsize=8)

fig.savefig(f'Figures/Fig_SI_DLA_DTM_validation.png')

# Misc

In [66]:
energy_data_with_J

,ene,err
a,,
0.05,-17.234867,0.000014
0.10,-17.235075,0.000013
0.15,-17.235396,0.000014
0.20,-17.235729,0.000014
0.25,-17.236109,0.000014
0.30,-17.236483,0.000013


In [67]:
print(energy_data_with_J.style.to_latex())

\begin{tabular}{lrr}
 & ene & err \\
a &  &  \\
0.050000 & -17.234867 & 0.000014 \\
0.100000 & -17.235075 & 0.000013 \\
0.150000 & -17.235396 & 0.000014 \\
0.200000 & -17.235729 & 0.000014 \\
0.250000 & -17.236109 & 0.000014 \\
0.300000 & -17.236483 & 0.000013 \\
\end{tabular}



In [68]:
energy_data_no_J

,ene,err
a,,
0.05,-17.235207,0.000042
0.10,-17.236848,0.000039
0.15,-17.239509,0.000041
0.20,-17.243531,0.000040
0.25,-17.248861,0.000040
0.30,-17.255505,0.000041


In [69]:
h2o_total_energy_validation['DLA']['CASINO']

,ene,err
tau,,
0.0025,-0.000732,0.000047
0.0050,-0.001335,0.000044
0.0100,-0.002307,0.000047
0.0200,-0.003543,0.000045
0.0400,-0.003759,0.000044
0.0800,0.001463,0.000046


In [70]:
df

wJ                  woJ                  DE          
            ene       err        ene       err       ene       err
a                                                                 
0.05 -17.234867  0.000014 -17.235207  0.000042  0.000340  0.000044
0.10 -17.235075  0.000013 -17.236848  0.000039  0.001773  0.000041
0.15 -17.235396  0.000014 -17.239509  0.000041  0.004113  0.000043
0.20 -17.235729  0.000014 -17.243531  0.000040  0.007802  0.000042
0.25 -17.236109  0.000014 -17.248861  0.000040  0.012752  0.000042
0.30 -17.236483  0.000013 -17.255505  0.000041  0.019022  0.000043

In [71]:
df['DE']

,ene,err
a,,
0.05,0.000340,0.000044
0.10,0.001773,0.000041
0.15,0.004113,0.000043
0.20,0.007802,0.000042
0.25,0.012752,0.000042
0.30,0.019022,0.000043
